# Module 4 — Production Monitoring, Observability, and Evaluation
**AnyCompany Bank: see inside the bank agent, alarm on it, and grade it, live on AWS in 20 minutes**

**Deck:** M04 · kernel **MLADAS (Python 3.12)** · **your AWS account** · `us-east-1` · Amazon Nova only

> **Start early: run §0 (down to "0.5 Ready check") 5–12 minutes before you go on to §1.** The online evaluation
> results for its warm-up conversations arrive ~10 minutes later, and §1's warm session idles out after 15 minutes.
> Read this module's `README.md` first: M04 needs CloudWatch Transaction Search turned on in your account.

> **The module in one line (slides 1, 55):** an HTTP 200 from an agent does not prove that the agent did its job.

<details><summary>Course, stack and cost</summary>

MLADAS — *Building Advanced Agentic Systems on AWS* (300-level). Strands Agents 1.57 · Amazon Bedrock AgentCore
(Runtime, Evaluations) · CloudWatch (Transaction Search, Logs Insights, metrics and alarms) · OpenTelemetry · Amazon Nova.
Cost: under \$0.50 per run in our runs (Nova tokens, Runtime, Evaluations, a few alarms and metrics); §4 prints this
run's bill.
</details>

> **Student copy.** Shared for learning; **not an official AWS training product**. AnyCompany Bank, the Octank Credit Bureau and every customer, card number, transaction and document in this notebook are **fictional**.
>
> Running it creates **billable AWS resources in your own AWS account** (us-east-1). Read this module's `README.md` first: set-up, expected results, cost and cleanup. The code finds your credentials through boto3's standard chain (environment variables, `AWS_PROFILE`, default profile, SSO); set `MLADAS_AWS_PROFILE` to pick a named profile. Code: MIT-0 (`LICENSE`) · text: CC BY-SA 4.0 (`LICENSE-CONTENT`).

### 0.1 What you will see
| Highlight | Slides | § |
|---|---|---|
| Session → trace → span for the bank agent on AgentCore Runtime and for a local agent; a tool fails but the request succeeds; head sampling, measured | 4, 8, 12–15, 21–26, 30, 31 | §1 |
| A looping agent trips two live CloudWatch alarms: loop detection and cost protection | 7, 27–29 | §2 |
| AgentCore Evaluations with Nova judges: on demand, then online | 36–48, 53–54 | §3 |
| What we measured, what it cost, cleanup verified by id | 6, 49 | §4 |

<details><summary>Deck vs 2026, at a glance (click to open)</summary>

Checked in a test account in us-east-1 on 2026-09-28. Each section shows its own box.

| Deck | Observed in Sept 2026 | Where |
|---|---|---|
| slide 25: spans in a `/aws/spans/default` log group; Transaction Search on by default | log group `aws/spans`; Transaction Search is a one-time setting per account and Region (see the README) | §1 |
| slides 21, 25: Runtime ADOT is automatic | direct code deploy needs `aws-opentelemetry-distro` + entry point `opentelemetry-instrument agent.py` | §0.4, §1 |
| slide 26: `OTel_EXPORTER_…` | only uppercase `OTEL_` is read | §1 |
| slide 29: alarm on `Duration` in `bedrock-agentcore` | no such metric; `AWS/Bedrock-AgentCore` `Duration` is per request | §2 |
| slide 39: 13 built-in evaluators | 18 built-in + 13 third-party | §3 |
| slides 43, 45, 47: `agentcore_evaluation` starter-toolkit code | no such package; boto3 or `bedrock_agentcore.evaluation` | §3 |
</details>

### 0.2 The cast and the vocabulary
The same fictional people as Modules 1–3: **James Chen** asks the bank agent on AgentCore Runtime about his card while
the card-risk service is down (§1); **Sofia Martinez**'s wire loops the wire assistant (§2) and she asks the savings
coach whether to invest (§3).

<details><summary>Who is who (click to open)</summary>

| Who | Id | In this module |
|---|---|---|
| Sofia Martinez | `CUST-1001`, checking `CHK-1001-01`, card …4417 | asks the savings coach whether to invest (§3); her wire `WIRE-4471` sets off the loop (§2) |
| James Chen | `CUST-1002`, card …7731 | asks the bank agent on Runtime for his balance, transactions and fraud score, while the card-risk service is down (§1) |
| The bank assistant | Nova 2 Lite, hosted on AgentCore Runtime (`bank_runtime_agent/`) | the agent we observe: ADOT traces it with no tracing code |
| The savings coach | the same hosted agent, persona `coach` / `coach_rule` | the bank's disclaimer rule, and a judge that checks it (§3) |
| The wire assistant | a local Nova 2 Lite agent | loops on a PENDING wire (§2) |
</details>

<details><summary>Vocabulary (click to open)</summary>

- **Three pillars** (slide 4): a **metric** shows *that* something is off, a **log** explains *why*, and a
  **trace** shows *where* in the request it happened.
- **Session ⊃ trace ⊃ span** (slide 8): a **session** is one whole conversation with a user, a **trace** is one
  request and its response inside that session, and a **span** is one timed step inside a trace (a model call, a
  tool call). Going from the session to a trace to a span is **progressive troubleshooting**.
- **OpenTelemetry (OTel)** (slides 21–23): the vendor-neutral standard for traces, metrics and logs; **OTLP** is its
  protocol. OTel instruments, collects and exports, but does not store or visualize. **ADOT** is the AWS Distro for
  OpenTelemetry. **Auto-instrumentation** adds spans without code changes.
- **Graceful degradation** (slide 14): the tool span fails, the trace still succeeds.
- **Head-based sampling** decides by rule when a trace begins; **tail-based sampling** decides once the trace has
  finished, for example to keep every trace with an error (slide 30).
- **Alert tiers** (slide 27): Tier 1 automated response · Tier 2 technical teams · Tier 3 business stakeholders.
  **Loop detection** and **cost protection alerts** (slide 28).
- **Evaluator** (slides 37–42): an LLM judge (or code) that scores agent behaviour at **session**, **trace** or
  **span** level (the API says `SESSION`, `TRACE`, `TOOL_CALL`). **Built-in** evaluators are `Builtin.<Name>`;
  **custom** evaluators carry your prompt, judge model and scale; **on-demand** vs **online** (slide 40).
</details>

## §0 — Setup and early start
Every cell in §0 is safe to re-run: it reuses what already exists instead of creating a second copy.

### 0.3 Environment and identity
One `RUN_ID` names and tags every AWS resource, so cleanup by name is exact. `make_agent()` always builds agents on
Amazon Nova, and a **ledger** records every model call's tokens and dollars plus the services' own charges; §4
totals it. The `OTEL_*` variables are set here because OpenTelemetry reads them once, before the first agent.

In [ ]:
import os, sys, json, re, time, uuid, warnings, logging
from datetime import datetime, timezone
from pathlib import Path

try:                                               # local Mac: python.org builds ship without trust roots
    import certifi
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except ImportError:
    pass

NOTEBOOK_DIR = Path.cwd()
DEMOS_ROOT = NOTEBOOK_DIR.parent                  # the repo root: shared helpers + bank data live here
if str(DEMOS_ROOT) not in sys.path:
    sys.path.insert(0, str(DEMOS_ROOT))

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*experimental.*")
for _noisy in ("strands", "httpx", "opentelemetry", "botocore.credentials"):
    logging.getLogger(_noisy).setLevel(logging.ERROR)

import importlib.metadata as _md
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
import mladas_common as mc
import mladas_viz as viz
import bank_data as bank
import agentcore_observability as obs
from mladas_common import LEDGER, make_agent, nova, text_of
from strands import tool

pd.set_option("display.html.use_mathjax", False)  # two '$' amounts in a table must not turn into LaTeX
pd.set_option("display.max_colwidth", 120)
viz.apply_style()

_REQUIRED = {"strands-agents": "1.57", "bedrock-agentcore": "1.23", "boto3": "1.43", "opentelemetry-sdk": "1.45",
             "opentelemetry-exporter-otlp-proto-http": "1.45"}
_versions = {**mc.stack_versions(), **{p: _md.version(p) for p in ("opentelemetry-sdk",
                                                                    "opentelemetry-exporter-otlp-proto-http")}}
for _pkg, _want in _REQUIRED.items():
    assert _versions[_pkg].startswith(_want), f"{_pkg} {_versions[_pkg]} found, {_want}.x expected — run ../setup_env.sh"
for _dep in ("jinja2", "requests", "opentelemetry.proto.collector.trace.v1.trace_service_pb2",
             "bedrock_agentcore.evaluation"):       # Styler tables, OTLP receiver + SigV4 export, Evaluations: fail here
    __import__(_dep)

_ident = mc.whoami()
SESSION = mc.get_session()                         # your AWS credentials, us-east-1: every AWS call in this notebook uses it
ACCOUNT, REGION = _ident["account"], _ident["region"]
# Re-running this cell keeps the same RUN_ID, so resources already created stay reachable for cleanup.
RUN_ID = globals().get("RUN_ID") or os.getenv("M04_RUN_ID") or \
    f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:4]}"
with open(NOTEBOOK_DIR / ".m04_run_ids", "a") as _run_ids:  # lets `teardown_m04.py --last` find this run later
    _run_ids.write(RUN_ID + "\n")
FORCE_FAIL = os.getenv("M04_FORCE_FAIL", "")       # fault injection: runtime | evaluations | alarms | otlp
DEV_STACK_RUN = obs.is_dev_stack(RUN_ID)           # a shared builder stack: adopted, never torn down by this notebook
OTEL_SETTINGS = obs.set_otel_env(RUN_ID)           # uppercase OTEL_* (slide 26's `OTel_` spelling is ignored)

print(f"AWS identity : {mc.mask_account(_ident['arn'], ACCOUNT)}  (profile={_ident['profile']}, region={REGION})")
print(f"RUN_ID       : {RUN_ID}" + ("  (from M04_RUN_ID: existing resources are adopted)" if os.getenv("M04_RUN_ID") else "")
      + ("  ⚠ shared DEV stack" if DEV_STACK_RUN else ""))
print("SDK stack    :", "  ".join(f"{k}={v}" for k, v in _versions.items()))
print("OTel env     :", "  ".join(f"{k}={v}" for k, v in OTEL_SETTINGS.items()))
if FORCE_FAIL:
    print(f"⚠ M04_FORCE_FAIL={FORCE_FAIL}: that dependency will fail on purpose so you can watch the fallback.")

### 0.4 Early start: the observed bank
The slow parts start **now, in the background**: the bank agent on **AgentCore Runtime** (then two warm-up
conversations), the **evaluators** and their **online evaluation**, the two **alarms**, and the local
**OpenTelemetry** pipeline. **While this runs (~70 s):** open the table below; then
check the `Ready:` line in 0.5.

<details><summary>What starts, and when it is ready (click to open)</summary>

| Lane | What starts now | Ready after |
|---|---|---|
| runtime → warm-up | the bank agent on AgentCore Runtime with ADOT (package → S3 → role → runtime) → §1's session warmed, plus two warm-up conversations (savings coach without and with the disclaimer rule, 2 turns each) | ~1 min |
| evaluators → online | three built-ins re-created with a Nova 2 Lite judge + one custom judge → an online evaluation of the Runtime agent (results ~10 min after the warm-ups) | ~5 s / after the runtime |
| alarms | one normal wire-agent session (the token baseline) → two 10-second CloudWatch alarms | ~5 s |
| telemetry | one OpenTelemetry pipeline for the local agents: in memory, an OTLP receiver in this process, CloudWatch | instant |
</details>

In [ ]:
import threading

M04_TAGS = {"project": "mladas", "module": "M04", "run_id": RUN_ID}   # on every resource §0 creates
DEPLOY_RUNTIME = os.getenv("M04_DEPLOY_RUNTIME", "1") == "1"        # the bank agent on AgentCore Runtime (§1, §3)
CREATE_EVALUATORS = os.getenv("M04_CREATE_EVALUATORS", "1") == "1"  # 3 derived + 1 custom Nova-judged evaluators (§3)
CREATE_ALARMS = os.getenv("M04_CREATE_ALARMS", "1") == "1"          # baseline session + 2 live alarms (§2)
ONLINE_EVAL_ON = os.getenv("M04_ONLINE_EVAL", "1") == "1"           # the online evaluation config (§3)
WARMUP_ON = os.getenv("M04_WARMUP", "1") == "1"                     # warm §1's session + the 2 warm-up conversations
BANK_AGENT_SOURCE = NOTEBOOK_DIR / "bank_runtime_agent"             # agent.py + requirements.txt (+ ../bank_data.py)
WARMUP_QUESTIONS = ("What's the balance in my checking account?",
                    "I have about $8,000 sitting in checking. Should I put some of it into an S&P 500 index fund, "
                    "or keep it in savings?")
print("Parts:", " · ".join(f"{k} {'on' if v else 'OFF'}" for k, v in (
    ("runtime", DEPLOY_RUNTIME), ("evaluators", CREATE_EVALUATORS), ("alarms", CREATE_ALARMS),
    ("online evaluation", ONLINE_EVAL_ON), ("warm-up", WARMUP_ON))))

In [ ]:
"""§0.4 exports (frozen 2026-09-28; read-only everywhere else, so a section that binds one of these names collides):
  Modules    obs (agentcore_observability, from s00) · threading
  Constants  M04_TAGS · DEPLOY_RUNTIME · CREATE_EVALUATORS · CREATE_ALARMS · ONLINE_EVAL_ON · WARMUP_ON ·
             BANK_AGENT_SOURCE · WARMUP_QUESTIONS (balance, S&P 500)
  Handles    OBS_STACK (ObservabilityBootstrap) ·
             BANK_RUNTIME (obs.RuntimeObsDeployment: .invoke(payload, session_id, trace_parent=, correlation_id=),
               .log_group, .service_name, .runtime_arn, .stop_session) · RUNTIME_SESSION_ID (§1's warmed session id) ·
             EVALUATORS (obs.EvaluatorSet: .ids, .describe(k), .evaluate_local(spans), .evaluate_runtime(sid, runtime, since)) ·
             ONLINE_EVALUATION (obs.OnlineEvaluation: .describe(), .results(session_ids), .batch(runtime, ids, sids);
               exists without a config when M04_ONLINE_EVAL=0, so .batch() still works) ·
             ALARMS (obs.AlarmSet: .names, .baseline_tokens, .wait_armed(), .wait_for("ALARM", since=), .history(k)) ·
             LOOP_METRICS (obs.LoopMetrics used by the baseline; §2 may attach it or a new one) ·
             LOCAL_OTEL (obs.LocalTelemetry, never None: .rows(sid), .readable_spans(sid), .force_flush(),
               .receiver, .receiver_ok, .cloudwatch_ok, .set_sampler) · OTLP_RECEIVER (obs.OtlpReceiver or None)
             Each handle is None when its part is switched off or forced to fail: call its wait_for_*() first.
  Filled in place by the lanes (read, never rebind):
             WARMUP_SESSIONS  [{persona, session_id, turns: [{prompt, answer, trace_id, wall_s, has_disclaimer,
                              tools, usage}], t_start, t_end, cloudwatch: {spans, events, traces, complete}, stopped}]
             ALARM_BASELINE   {tokens, tool_calls, stop_reason, session_id, wall_s, cost_usd, adopted}
             OBS_T0 (epoch s when §0.4 started)
  Waits      wait_for_runtime · wait_for_warmup · wait_for_evaluators · wait_for_online_eval · wait_for_alarms
             (max_wait s, <= 90 in the demo) -> bool: one updating status line, one ⚠ line when unavailable, never raise
  Cleanup    observability_cleanup_steps() -> [(label, fn)] for mc.Cleanup().start(...) · OBS_CLEANUP_STATUS
             {resource: status} · observability_verify_gone(sweep=True) -> {resource: "gone ✓" | "NOT CLEANED: …"} ·
             observability_resources() -> [{lane, resource, name, id}] · observability_cost_rows() -> [{service, units,
             unit_price, usd, note}] (book with LEDGER.add_service; on-demand evaluations and Runtime tokens are ledger
             rows already)
  Ledger     "§0 baseline" (the alarm baseline session) · "§0 warmup" (Runtime warm-up turns, booked from the agent's usage)
"""
from typing import Any
from botocore.exceptions import BotoCoreError, ClientError


class ObservabilityBootstrap:
    """§0.4's early start. Lanes run in background threads and never raise:
      telemetry            one global TracerProvider (in-memory + OTLP receiver + CloudWatch), in the foreground
      runtime -> warmup    deploy (or adopt) the ADOT bank agent -> READY -> warm §1's session (no model call) ->
                           2 warm-up conversations (coach, coach_rule; 2 turns each, in parallel) -> stop them
      evaluators -> online derived Helpfulness / GoalSuccessRate / ToolSelectionAccuracy + custom disclaimer judge (Nova
                           2 Lite) -> online evaluation config on the runtime's log group once the runtime is READY
      alarms               baseline wire-agent session (tokens) -> loop + cost alarms (10 s, RunId dimension)
    start() is idempotent: running or READY lanes are left alone, FAILED lanes retry (every step adopts this RUN_ID's
    resources). Handles are None when a lane is switched off (flag) or forced to fail (M04_FORCE_FAIL)."""

    LANES = ("telemetry", "runtime", "warmup", "evaluators", "online", "alarms")
    LABELS = {"telemetry": "local OpenTelemetry pipeline", "runtime": "bank agent on AgentCore Runtime (ADOT)",
              "warmup": "warm-up conversations", "evaluators": "Nova-judged evaluators",
              "online": "online evaluation config", "alarms": "loop + cost alarms"}
    USED = {"telemetry": "§1.3", "runtime": "§1, §3", "warmup": "§1, §3 (online results)",
            "evaluators": "§3", "online": "§3.4", "alarms": "§2"}
    TYPICAL = {"telemetry": "instant", "runtime": "~1-1.5 min", "warmup": "~20 s after the runtime",
               "evaluators": "~5 s", "online": "~2 s after the runtime", "alarms": "~15 s"}

    def __init__(self, run_id: str, session, *, runtime: bool = True, evaluators: bool = True, alarms: bool = True,
                 online: bool = True, warmup: bool = True, force_fail: str = ""):
        self.run_id, self.session, self.t0, self.force_fail = run_id, session, time.time(), force_fail
        self.state = {lane: "NOT_STARTED" for lane in self.LANES}
        self.off: dict[str, str] = {}
        self.errors: dict[str, str] = {}
        self.ready_at: dict[str, float] = {}
        self.events: list[tuple[float, str, str]] = []
        self._threads: dict[str, threading.Thread] = {}
        self._stopping = threading.Event()
        self.runtime = self.evaluators = self.online = self.alarms = self.loop_metrics = self.local = None
        self.demo_session_id: str | None = None
        self.warmups: list[dict] = []
        self.baseline: dict = {}
        self.probe: dict = {}
        tags = dict(M04_TAGS)

        if not runtime:
            self._switch_off(("runtime", "warmup", "online"), "M04_DEPLOY_RUNTIME=0")
        elif force_fail == "runtime":
            self._fail(("runtime", "warmup", "online"), "forced by M04_FORCE_FAIL=runtime")
        else:
            try:
                self.runtime = obs.RuntimeObsDeployment(run_id, session, BANK_AGENT_SOURCE, tags=tags,
                                                        extra_files=[DEMOS_ROOT / "bank_data.py"])
                self.demo_session_id = obs.runtime_session_id("demo", run_id)
            except Exception as e:  # noqa: BLE001 — setup never raises
                self._fail(("runtime", "warmup", "online"), self._why(e))
            if not warmup and self.runtime is not None:
                self._switch_off(("warmup",), "M04_WARMUP=0")

        if not evaluators:
            self._switch_off(("evaluators",), "M04_CREATE_EVALUATORS=0")
            if "online" not in self.off and "online" not in self.errors:
                self._switch_off(("online",), "needs the evaluators (M04_CREATE_EVALUATORS=0)")
        elif force_fail == "evaluations":
            self._fail(("evaluators",), "forced by M04_FORCE_FAIL=evaluations")
            if "online" not in self.off and "online" not in self.errors:
                self._fail(("online",), "needs the evaluators (forced by M04_FORCE_FAIL=evaluations)")
        else:
            self.evaluators = obs.EvaluatorSet(session, run_id, tags=tags)
        if self.runtime is not None and self.evaluators is not None:
            self.online = obs.OnlineEvaluation(session, run_id, tags=tags)
            if not online:
                self._switch_off(("online",), "M04_ONLINE_EVAL=0 (batch evaluation still available)")

        if not alarms:
            self._switch_off(("alarms",), "M04_CREATE_ALARMS=0")
        elif force_fail == "alarms":
            self._fail(("alarms",), "forced by M04_FORCE_FAIL=alarms")
        else:
            self.alarms = obs.AlarmSet(session, run_id, tags=tags)
            self.loop_metrics = obs.LoopMetrics(session, run_id)

    # ------------------------------------------------------------------------------------------ bookkeeping
    @staticmethod
    def _why(e: BaseException) -> str:
        return mc.short(mc.mask_account(f"{type(e).__name__}: {e}", ACCOUNT), 300)

    def _event(self, lane: str, step: str, msg: str = "") -> None:
        self.events.append((round(time.time() - self.t0, 1), f"{lane} {step}", mc.mask_account(msg, ACCOUNT)))

    def _set(self, lane: str, state: str, msg: str = "") -> None:
        self.state[lane] = state
        if state == "READY":
            self.ready_at.setdefault(lane, round(time.time() - self.t0, 1))
            self.errors.pop(lane, None)
        self._event(lane, state, msg)

    def _switch_off(self, lanes, why: str) -> None:
        for lane in lanes:
            self.off[lane], self.state[lane] = why, "OFF"

    def _fail(self, lanes, why: str) -> None:
        for lane in lanes:
            self.errors[lane] = why
            self._set(lane, "FAILED", why)

    # ------------------------------------------------------------------------------------------ lanes
    def start(self) -> "ObservabilityBootstrap":
        """Start every lane that is not running or READY yet. Returns at once (telemetry is set up in the foreground)."""
        if self._stopping.is_set():
            return self
        if self.local is None:
            self._set("telemetry", "STARTING")
            self.local = obs.LocalTelemetry.setup(self.session, run_id=self.run_id, force_fail=self.force_fail)
            st = self.local.status
            self._set("telemetry", "READY", f"receiver {st.get('receiver')} · CloudWatch {st.get('cloudwatch')}")
        if self.runtime is not None:
            self._launch("runtime", self._runtime_lane)
        if self.evaluators is not None:
            self._launch("evaluators", self._evaluators_lane)
        if self.online is not None and "online" not in self.off:
            self._launch("online", self._online_lane)
        if self.alarms is not None:
            self._launch("alarms", self._alarms_lane)
        return self

    def _launch(self, lane: str, fn) -> None:
        running = self._threads.get(lane)
        if (running and running.is_alive()) or self.state[lane] == "READY":
            return
        self._threads[lane] = threading.Thread(target=self._guard, args=(lane, fn), name=f"m04-{lane}", daemon=True)
        self._threads[lane].start()

    def _guard(self, lane: str, fn) -> None:
        """Run a lane; retry once after 5 s (every step adopts what the first try created); never raise."""
        for attempt in (1, 2):
            try:
                fn()
                return
            except BaseException as e:  # noqa: BLE001 — a failed early start must never stop Run All
                why = self._why(e)
                if attempt == 1 and not self._stopping.is_set():
                    self._event(lane, "RETRYING", why)
                    time.sleep(5)
                    continue
                self._fail((lane,), why)
                if lane == "runtime":
                    deps = ("warmup", "online") if not self.runtime.ready else ("warmup",)
                    for dep in deps:
                        if self.state[dep] not in ("OFF", "READY"):
                            self._fail((dep,), "needs the runtime")

    def _runtime_lane(self) -> None:
        rt = self.runtime
        if self.state["runtime"] != "READY":
            self._set("runtime", "ADOPTING", f"looking for {rt.name}")
            if not rt.adopt():
                self._set("runtime", "DEPLOYING", f"{rt.name}: package (linux/arm64 wheels) -> S3 -> role -> "
                                                  "CreateAgentRuntime (entryPoint opentelemetry-instrument agent.py)")
                rt.start()
            if not rt.wait(timeout=600, poll=2):
                raise RuntimeError(f"runtime {rt.status}: {rt.error or 'not READY after 600 s'}")
            self._set("runtime", "PROBING", "first request of §1's session (cold start, no model call)")
            self._warm_demo_session()                 # health probe: READY is not enough, it must answer
            self._set("runtime", "READY", f"{rt.runtime_id} ({'adopted' if rt.adopted else 'new'}) · §1's session "
                                          f"warm after {self.probe.get('wall_s')} s ({self.probe.get('attempts')} attempt(s))")
        if "warmup" in self.off or self.state["warmup"] == "READY" or self._stopping.is_set():
            return
        self._set("warmup", "RUNNING", "coach / coach_rule x 2 turns, in parallel")
        try:
            self._warmup()
        except Exception as e:  # noqa: BLE001 — a failed warm-up only affects the online results, never the runtime
            self._fail(("warmup",), self._why(e))

    def _warm_demo_session(self) -> None:
        """{"warmup": true} on §1's session: starts its microVM (5-8 s that no span shows). A brand-new runtime's first
        request once failed with "Runtime initialization time exceeded" (then 502 on a retry 5 s later; 1 of 2 fresh
        runs, 2026-09-28) -> up to 3 attempts, each after StopRuntimeSession (a new microVM for the same session id)."""
        rt, err = self.runtime, None
        for attempt in (1, 2, 3):
            with LEDGER.local_scope("§0 warmup"):
                body = rt.invoke({"warmup": True}, self.demo_session_id, correlation_id="warmup-demo", book=False)
            if not body.get("error"):
                self.probe.update({"wall_s": body.get("wall_s"), "attempts": attempt})
                return
            err = body["error"]
            self._event("runtime", "PROBE_RETRY", f"attempt {attempt}: {err}")
            if attempt < 3:
                rt.stop_session(self.demo_session_id)
                time.sleep(10)
        raise RuntimeError(f"READY, but §1's session did not answer 3 times: {err}")

    def _warmup(self) -> None:
        rt = self.runtime
        self.warmups.clear()

        def ask(persona: str, sid: str, q: str, i: int) -> dict:
            for attempt in (1, 2):
                with LEDGER.local_scope("§0 warmup"):
                    body = rt.invoke({"prompt": q, "customer_id": "CUST-1001", "persona": persona}, sid,
                                     correlation_id=f"warmup-{persona}-{i}")
                if not body.get("error") or attempt == 2:
                    return body
                self._event("warmup", "RETRY", f"{persona} turn {i + 1}: {body['error']}")
                if i == 0:
                    rt.stop_session(sid)               # a fresh microVM for the first turn
                time.sleep(5)
            return body

        def conversation(persona: str) -> None:
            sid = obs.runtime_session_id(persona, self.run_id)
            rec = {"persona": persona, "session_id": sid, "turns": [], "t_start": time.time(), "t_end": None,
                   "stopped": False}
            self.warmups.append(rec)
            for i, q in enumerate(WARMUP_QUESTIONS):
                body = ask(persona, sid, q, i)
                rec["turns"].append({"prompt": q, "answer": body.get("answer"), "error": body.get("error"),
                                     "trace_id": body.get("trace_id"), "wall_s": body.get("wall_s"),
                                     "tools": body.get("tools"), "usage": body.get("usage"),
                                     "has_disclaimer": obs.has_disclaimer(body.get("answer") or "")})
            rec["t_end"] = time.time()
            # Stop the microVM only once both traces (spans + content records) are in CloudWatch: stopping right after
            # the answer loses the last request's spans (ADOT exports in batches). Then GB-hours stop accruing.
            _docs, rec["cloudwatch"] = obs.runtime_session_docs(sid, rt.log_group, session=self.session,
                                                                since=rec["t_start"] - 60, min_traces=len(WARMUP_QUESTIONS),
                                                                max_wait=90, quiet=True)
            rec["stopped"] = rt.stop_session(sid)

        threads = [threading.Thread(target=conversation, args=(p,), daemon=True) for p in ("coach", "coach_rule")]
        for t in threads:
            t.start()
        for t in threads:
            t.join(timeout=240)
        bad = [f"{w['persona']}: {t['error']}" for w in self.warmups for t in w["turns"] if t.get("error")]
        bad += [f"{w['persona']}: spans not complete in CloudWatch ({w.get('cloudwatch')})" for w in self.warmups
                if not (w.get("cloudwatch") or {}).get("complete")]
        if bad or len(self.warmups) < 2 or any(len(w["turns"]) < 2 for w in self.warmups):
            raise RuntimeError("warm-up conversations incomplete: " + ("; ".join(bad) or "a thread did not finish"))
        self._set("warmup", "READY", " · ".join(
            f"{w['persona']} {sum(t['wall_s'] or 0 for t in w['turns']):.1f} s (+{w['cloudwatch']['seconds']} s until "
            f"its {w['cloudwatch']['traces']} traces were in CloudWatch)" for w in self.warmups))

    def _evaluators_lane(self) -> None:
        ev = self.evaluators
        self._set("evaluators", "CREATING", " · ".join(ev.names.values()))
        if not ev.create():
            raise RuntimeError(ev.error or "evaluators not created")
        self._set("evaluators", "READY", f"{len(ev.ids)} ACTIVE" + (f" (adopted: {', '.join(ev.adopted)})"
                                                                     if ev.adopted else ""))

    def _online_lane(self) -> None:
        on = self.online
        self._set("online", "ROLE", on.role_name)
        if not on.adopt() and not on.role_arn and not on.create_role():
            raise RuntimeError(on.error or "execution role not created")
        self._set("online", "WAITING", "for the runtime (its log group) and the evaluators")
        deadline = time.time() + 900
        while not (self.runtime.ready and self.state["evaluators"] == "READY"):   # deploy READY: its log group exists
            if self._stopping.is_set() or time.time() > deadline or \
                    self.state["runtime"] in ("FAILED", "OFF") or self.state["evaluators"] in ("FAILED", "OFF"):
                raise RuntimeError("the runtime or the evaluators are not available")
            time.sleep(2)
        self._set("online", "CREATING", f"{on.config_name}: sampling 100 %, session idle timeout 1 min")
        ids = [self.evaluators.ids[k] for k in ("helpful", "goal", "disclaimer")]
        if not on.create(self.runtime, ids):
            raise RuntimeError(on.error or "online evaluation config not ACTIVE")
        self._set("online", "READY", f"{on.config_id} ACTIVE" + (" (adopted)" if on.adopted else ""))

    def _alarms_lane(self) -> None:
        al = self.alarms
        self._set("alarms", "ADOPTING", " · ".join(al.names.values()))
        if al.adopt():
            self.baseline.update({"tokens": al.baseline_tokens, "adopted": True})
            self._set("alarms", "READY", f"adopted (baseline {al.baseline_tokens:,.0f} tokens)")
            return
        self._set("alarms", "BASELINE", "one normal wire-agent session (the tool answers COMPLETED)")
        desk = obs.WireDesk("COMPLETED")
        sid = obs.runtime_session_id("baseline", self.run_id)
        agent = obs.wire_agent(desk, hooks=[self.loop_metrics], trace_attributes={"session.id": sid})
        t = time.time()
        with LEDGER.local_scope("§0 baseline"):
            res = agent(obs.WIRE_QUESTION, invocation_state={"correlation_id": "alarm-baseline"})
        inv = res.metrics.latest_agent_invocation
        tokens = int(inv.usage.get("totalTokens", 0)) if inv else 0
        if not tokens:
            raise RuntimeError("baseline session reported 0 tokens")
        self.baseline.update({"tokens": tokens, "tool_calls": desk.calls, "stop_reason": res.stop_reason,
                              "session_id": sid, "wall_s": round(time.time() - t, 2), "adopted": False,
                              "cost_usd": round(mc.cost_usd(obs.RUNTIME_MODEL_ID, dict(inv.usage)), 6)})
        self._set("alarms", "CREATING", f"loop: ToolCallsInSession > {obs.LOOP_MAX_TOOL_CALLS} · cost: tokens / "
                                        f"{tokens:,} > {obs.COST_MAX_RATIO} (10-s periods, ActionsEnabled=False)")
        if not al.create(tokens):
            raise RuntimeError(al.error or "alarms not created")
        self._set("alarms", "READY", f"baseline {tokens:,} tokens, {desk.calls} tool call(s); first evaluation "
                                     "within ~1 min")

    # ------------------------------------------------------------------------------------------ status
    def lane_state(self, lane: str) -> str:
        return self.state.get(lane, "UNKNOWN")

    def describe(self, lane: str) -> str:
        st = self.lane_state(lane)
        if lane == "runtime" and self.runtime is not None and st not in ("READY", "FAILED", "OFF"):
            return f"{st} ({self.runtime.status})"
        if lane == "telemetry" and self.local is not None:
            return (f"{st} (in-memory on · receiver {'on' if self.local.receiver_ok else 'OFF'} · "
                    f"CloudWatch {'on' if self.local.cloudwatch_ok else 'OFF'})")
        return st

    def error(self, lane: str) -> str:
        err = self.errors.get(lane) or (self.runtime.error if lane == "runtime" and self.runtime is not None else None)
        return mc.short(mc.mask_account(str(err or self.describe(lane)), ACCOUNT), 220)

    def ready(self, lane: str) -> bool:
        return self.lane_state(lane) == "READY"

    def wait(self, lane: str, max_wait: float = 90) -> bool:
        """True once `lane` is READY. One updating status line while waiting; one ⚠ line if it is off, failed or
        still building after max_wait. Never raises."""
        label = self.LABELS[lane]
        if self.ready(lane):
            return True
        if lane in self.off:
            print(f"⚠ No {label} in this run ({self.off[lane]}) — cells that need it print this line and skip their "
                  "live calls.")
            return False
        if self.lane_state(lane) != "FAILED":
            mc.wait_until(lambda: self.lane_state(lane) in ("READY", "FAILED"), max_wait=max_wait, label=label,
                          poll=2, describe=lambda _v: self.describe(lane))
        if self.ready(lane):
            return True
        if self.lane_state(lane) == "FAILED":
            print(f"⚠ {label}: not available ({self.error(lane)}) — cells that need it print this line and skip "
                  "their live calls.")
        else:
            print(f"⚠ {label}: not ready yet ({self.describe(lane)}, {time.time() - self.t0:.0f} s after §0 started "
                  f"it; usually {self.TYPICAL[lane]}) — re-run this cell in a minute.")
        return False

    def print_lanes(self) -> None:
        print(f"Early start of RUN_ID {self.run_id} (background; §0.5 waits for it):")
        for lane in self.LANES:
            state = f"OFF ({self.off[lane]})" if lane in self.off else (
                f"FAILED ({self.errors[lane]})" if lane in self.errors else self.describe(lane))
            print(f"  {self.LABELS[lane]:<40} {mc.short(state, 60):<60} used in {self.USED[lane]}")

    def progress(self) -> list[tuple[float, str, str]]:
        return list(self.events)

    @property
    def status(self) -> str:
        return " · ".join(f"{lane} {self.lane_state(lane)}" for lane in self.LANES if lane not in self.off)

    def online_results_eta(self) -> float | None:
        """Epoch s when the warm-ups' online results should be in (~10 min after the last warm-up turn; 6-11 min)."""
        ends = [w["t_end"] for w in self.warmups if w.get("t_end")]
        return max(ends) + 600 if ends and self.ready("online") else None

    # ------------------------------------------------------------------------------------------ resources
    def resources(self) -> list[dict]:
        rows = []

        def add(lane, resource, name, rid=None):
            rows.append({"lane": lane, "resource": resource, "name": name,
                         "id": mc.mask_account(rid, ACCOUNT) if rid else ""})
        n = obs.resource_names(self.run_id)
        if self.runtime is not None:
            add("runtime", "AgentCore Runtime (HTTP, ADOT)", self.runtime.name, self.runtime.runtime_id)
            add("runtime", "IAM execution role", self.runtime.role_name)
            add("runtime", "S3 bucket (code.zip)", self.runtime.bucket)
            add("runtime", "CloudWatch log group", self.runtime.log_group or f"{n['runtime_log_group_prefix']}-…")
        if self.evaluators is not None:
            for k, name in self.evaluators.names.items():
                add("evaluators", f"evaluator ({obs.EVALUATOR_SPECS[k]['kind']}, {obs.EVALUATOR_SPECS[k]['level']})",
                    name, self.evaluators.ids.get(k))
        if self.online is not None:
            add("online", "IAM execution role", self.online.role_name)
            if "online" not in self.off:
                add("online", "online evaluation config", self.online.config_name, self.online.config_id)
                add("online", "results log group", self.online.results_log_group or f"{n['results_log_group_prefix']}-…")
        if self.alarms is not None:
            for k, name in self.alarms.names.items():
                add("alarms", f"CloudWatch alarm ({k}, 10 s)", name)
            add("alarms", "custom metrics (expire, cannot be deleted)", f"{obs.M04_NAMESPACE} ToolCallsInSession, "
                                                                        "TokensInSession")
        return rows

    # ------------------------------------------------------------------------------------------ cleanup contract
    def _join(self, timeout: float, *lanes: str) -> None:
        deadline = time.time() + timeout
        for lane, t in list(self._threads.items()):
            if t.is_alive() and (not lanes or lane in lanes):
                t.join(max(0.0, deadline - time.time()))

    def _step(self, label: str, fn):
        def run() -> list[str]:
            out = {mc.mask_account(k, ACCOUNT): mc.mask_account(v, ACCOUNT) for k, v in dict(fn() or {}).items()}
            OBS_CLEANUP_STATUS.update(out)
            bad = [k for k, v in out.items() if v.startswith("NOT CLEANED") or v.startswith("error")]
            if bad:
                raise RuntimeError("; ".join(f"{k} -> {out[k]}" for k in bad))
            return [f"{k}: {v}" for k, v in out.items()] or [f"{label}: nothing to delete"]
        return label, run

    def _clean_online(self) -> dict:
        self._join(120, "online")
        return self.online.teardown()

    def _clean_evaluators(self) -> dict:
        self._join(60, "evaluators", "online")
        return self.evaluators.teardown()

    def _clean_alarms(self) -> dict:
        self._join(90, "alarms")
        return self.alarms.teardown()

    def _clean_runtime(self) -> dict:
        self._join(240, "runtime")                    # a still-running deploy or warm-up finishes first
        return self.runtime.teardown(wait=True)

    def cleanup_steps(self) -> list[tuple[str, Any]]:
        if DEV_STACK_RUN and os.getenv("M04_ALLOW_DEV_TEARDOWN") != "1":
            print(f"⚠ RUN_ID {self.run_id} is a shared dev stack: cleanup skipped.")
            return []
        self._stopping.set()                          # lanes stop at their next step; no retries from now on
        steps = []
        if self.online is not None:
            steps.append(self._step("online evaluation: config, results log group, batch jobs, role", self._clean_online))
        if self.evaluators is not None:
            steps.append(self._step("evaluators (after the online config that locks them)", self._clean_evaluators))
        if self.alarms is not None:
            steps.append(self._step("CloudWatch alarms", self._clean_alarms))
        if self.runtime is not None:
            steps.append(self._step("AgentCore Runtime: sessions, runtime, log groups, role, bucket", self._clean_runtime))
        if self.local is not None:
            steps.append(self._step("local telemetry: flush, shut down, stop the OTLP receiver", self.local.shutdown))
        return steps

    def verify_gone(self, sweep: bool = True) -> dict[str, str]:
        """Every resource BY ID (get_* -> NotFound or DELETING), then (sweep=True) anything else with this run's names."""
        out: dict[str, str] = {}
        for label, fn in (("runtime", lambda: self.runtime.verify_gone(self.runtime.log_group)),
                          ("evaluators", lambda: self.evaluators.verify_gone()),
                          ("online", lambda: self.online.verify_gone()),
                          ("alarms", lambda: self.alarms.verify_gone())):
            handle = {"runtime": self.runtime, "evaluators": self.evaluators, "online": self.online,
                      "alarms": self.alarms}[label]
            if handle is None:
                continue
            try:
                out.update(fn())
            except Exception as e:  # noqa: BLE001
                out[f"{label} verification"] = f"NOT CLEANED: check failed ({type(e).__name__})"
        if sweep:
            try:
                for kind, items in obs.find_run_resources(self.session, self.run_id).items():
                    for item in items:
                        if isinstance(item, dict) and item.get("status") == "DELETING":
                            continue
                        name = item if isinstance(item, str) else item.get("name") or item.get("id")
                        out.setdefault(f"{kind}: {name} (found by name)", "NOT CLEANED: still exists")
            except Exception as e:  # noqa: BLE001
                out["name sweep"] = f"NOT CLEANED: sweep failed ({type(e).__name__})"
        return {mc.mask_account(k, ACCOUNT): mc.mask_account(v, ACCOUNT) for k, v in out.items()}


def wait_for_runtime(max_wait: float = 90) -> bool:
    """True once the bank agent on AgentCore Runtime is READY (ADOT auto-instrumented). One status line; never raises."""
    return OBS_STACK.wait("runtime", max_wait)


def wait_for_warmup(max_wait: float = 90) -> bool:
    """True once §1's Runtime session is warm and both warm-up conversations are done (WARMUP_SESSIONS is filled)."""
    return OBS_STACK.wait("warmup", max_wait)


def wait_for_evaluators(max_wait: float = 90) -> bool:
    """True once the 4 Nova-judged evaluators are ACTIVE (EVALUATORS.ids has helpful, goal, toolsel, disclaimer)."""
    return OBS_STACK.wait("evaluators", max_wait)


def wait_for_online_eval(max_wait: float = 90) -> bool:
    """True once the online evaluation config is ACTIVE on the runtime's log group (results come ~10 min later)."""
    return OBS_STACK.wait("online", max_wait)


def wait_for_alarms(max_wait: float = 90) -> bool:
    """True once the loop + cost alarms exist (baseline measured). Before the loop, also call ALARMS.wait_armed()."""
    return OBS_STACK.wait("alarms", max_wait)


def observability_resources() -> list[dict]:
    """[{lane, resource, name, id}] for everything §0 created or adopted (ids account-masked)."""
    return OBS_STACK.resources()


def observability_cost_rows() -> list[dict]:
    """This run's non-token charges as rows {service, units, unit_price, usd, note} (estimates labelled): Runtime
    vCPU/GB-hours, online/batch evaluation fees + estimated judge tokens, alarms, custom metrics, PutMetricData,
    span ingestion, Logs Insights scans. Book each with LEDGER.add_service(...)."""
    return obs.service_cost_rows(runtime=OBS_STACK.runtime, evaluators=OBS_STACK.evaluators, online=OBS_STACK.online,
                                 alarms=OBS_STACK.alarms, loop_metrics=obs.LoopMetrics.instances, local=OBS_STACK.local,
                                 since=OBS_STACK.t0 - 60)


OBS_CLEANUP_STATUS: dict[str, str] = {}   # filled by the cleanup steps: resource -> "gone ✓" | "NOT CLEANED: …"


def observability_cleanup_steps() -> list[tuple[str, Any]]:
    """The wrap-up's cleanup contract: [(label, fn)] for mc.Cleanup().start(...), in dependency order — the online
    evaluation config first (it locks its evaluators; its results log group and role go with it, and the aws/spans index
    policy it added is reverted when safe), the evaluators, the alarms, the Runtime (live sessions stopped first; its log
    groups, role and bucket), then the local telemetry. Each fn records every resource in OBS_CLEANUP_STATUS and raises
    at the end if anything is not gone, so Cleanup.problems names it. Returns [] (and prints ⚠) for a dev stack."""
    return OBS_STACK.cleanup_steps()


def observability_verify_gone(sweep: bool = True) -> dict[str, str]:
    """By-id verification of every §0 resource (+ a name sweep for leftovers): {resource: "gone ✓" | "NOT CLEANED: …"}."""
    return OBS_STACK.verify_gone(sweep)

In [ ]:
OBS_STACK = globals().get("OBS_STACK")                    # re-running this cell reuses the live jobs
if OBS_STACK is None:
    OBS_STACK = ObservabilityBootstrap(RUN_ID, SESSION, runtime=DEPLOY_RUNTIME, evaluators=CREATE_EVALUATORS,
                                       alarms=CREATE_ALARMS, online=ONLINE_EVAL_ON, warmup=WARMUP_ON,
                                       force_fail=FORCE_FAIL)
OBS_STACK.start()                                         # background lanes; a FAILED lane retries and adopts by RUN_ID

# The handles every later section uses (None when a part is switched off or forced to fail: call its wait_for_*()).
BANK_RUNTIME = OBS_STACK.runtime            # obs.RuntimeObsDeployment: the bank agent on AgentCore Runtime (§1, §3)
RUNTIME_SESSION_ID = OBS_STACK.demo_session_id   # §1's session on the Runtime, warmed in the background
EVALUATORS = OBS_STACK.evaluators           # obs.EvaluatorSet: helpful · goal · toolsel · disclaimer (§3)
ONLINE_EVALUATION = OBS_STACK.online        # obs.OnlineEvaluation: the online config (+ batch fallback) (§3)
ALARMS = OBS_STACK.alarms                   # obs.AlarmSet: loop detection + cost protection (§2)
LOOP_METRICS = OBS_STACK.loop_metrics       # obs.LoopMetrics hook: PutMetricData per tool call (§2)
LOCAL_OTEL = OBS_STACK.local                # obs.LocalTelemetry: in-memory + OTLP receiver + CloudWatch (§1, §3)
OTLP_RECEIVER = LOCAL_OTEL.receiver         # obs.OtlpReceiver on 127.0.0.1 (the "enterprise backend") or None
WARMUP_SESSIONS = OBS_STACK.warmups         # filled in place by the warm-up lane (§3)
ALARM_BASELINE = OBS_STACK.baseline         # filled in place by the alarms lane (§2)
OBS_T0 = OBS_STACK.t0                       # when §0.4 started
OBS_STACK.print_lanes()

### 0.5 Ready check
One status line per part; the cell returns when all are READY (on a re-run, at once). It also says when the online
evaluation results for the warm-up conversations should be in.

In [ ]:
_ready_deadline = OBS_T0 + 270              # keeps the whole setup inside its 5-minute lead on a fresh RUN_ID
_ready = {lane: fn(max_wait=max(5.0, _ready_deadline - time.time())) for lane, fn in (
    ("runtime", wait_for_runtime), ("warmup", wait_for_warmup), ("evaluators", wait_for_evaluators),
    ("online", wait_for_online_eval), ("alarms", wait_for_alarms))}
_down = [lane for lane, ok in _ready.items() if not ok and (lane in OBS_STACK.off or OBS_STACK.lane_state(lane) == "FAILED")]
_building = [lane for lane, ok in _ready.items() if not ok and lane not in _down]
_eta = OBS_STACK.online_results_eta()
print(f"Ready: {sum(_ready.values())}/{len(_ready)} parts READY · RUN_ID {RUN_ID} · "
      f"{time.time() - OBS_T0:.0f} s after the early start"
      + (f" · online results expected in ~{max(0, (_eta - time.time()) / 60):.0f} min (usually 6-11 min after the "
         "warm-ups)" if _eta else "")
      + (f" · still building: {', '.join(_building)} (re-run this cell in a minute)" if _building else "")
      + (f" · OFF or FAILED: {', '.join(_down)} (see this module's README.md)" if _down else ""))
print("Telemetry:", " · ".join(f"{k} {v}" for k, v in LOCAL_OTEL.status.items() if k in ("receiver", "cloudwatch")))
if DEV_STACK_RUN:
    print(f"⚠ {RUN_ID} is a shared DEV stack, meant for development only (unset M04_RUN_ID).")

## §1 — Session, trace, span: inside the bank agent
A metric says that something is off, a log says why and a trace says where, so we start with the traces of one
bank conversation. *Deck slides 4, 8–15, 18, 21–26, 30–31.*

James Chen asks the bank agent on **AgentCore Runtime** three questions in one session while the card-risk service
is down. We read the telemetry back from CloudWatch and walk it the deck's way: **progressive troubleshooting means
going from the session to one trace to one span** (slide 8). Then a local agent gets the same view, and we measure
what head sampling would throw away.

| We say… | only when this check, printed in the same cell, says so |
|---|---|
| a span **failed** | its status is `ERROR` in the spans read back from CloudWatch or from memory |
| the request **succeeded** | the Runtime returned an answer and no span above the failed one is `ERROR` |
| the answer **degraded gracefully** | it names the unavailable check and states no fraud score |
| a destination got the **same** spans | its span ids equal the in-memory copy's |

In [ ]:
LEDGER.clear_section("§1")                          # re-running §1 must not double count in §4
S01_RUNTIME_OK = wait_for_runtime(max_wait=60)      # False (+ one ⚠ line) when the Runtime is off or failed
S01_JAMES = [                                       # James Chen (CUST-1002, card 7731): one conversation, 3 requests
    {"prompt": "What is the balance of my checking account?", "customer_id": "CUST-1002"},
    {"prompt": "Show me the recent transactions on my card ending 7731.", "customer_id": "CUST-1002"},
    {"prompt": "Is my card ending 7731 at risk of fraud? Check its fraud score.", "customer_id": "CUST-1002",
     "simulate_outage": "timeout"},                 # the card-risk service hangs 1.2 s, then TimeoutError (slide 14)
]

In [ ]:
# Plumbing (not teaching): bounded waits with one status line, the evidence tables, the two charts, and the Runtime trace
# recorded in research, shown only when this run has no Runtime (M04_DEPLOY_RUNTIME=0 or M04_FORCE_FAIL=runtime).
S01_UNAVAILABLE = re.compile(r"unavailable|not available|timed? ?out|could not|couldn't|unable|try (?:it |this |that )?again", re.I)
S01_SCORE_STATED = re.compile(r"score\s*(?:is|of|was|:|=)?\s*\d{1,3}\b", re.I)
S01_TRACING_CALLS = re.compile(r"start_as_current_span|start_span|TracerProvider|SpanExporter|SpanProcessor|"
                               r"StrandsTelemetry|set_tracer_provider")
S01_RECORDED = {      # recorded in a test account in us-east-1 on 2026-09-28 (the course team's runs)
    "1.1": ("TRACE 3 6ab9ea7e…  (recorded in a test account, 2026-09-28)\n"
            "  POST /invocations                            +  2.71s   2.37s\n"
            "    invoke_agent bank_assistant                +  2.71s   2.36s\n"
            "      execute_event_loop_cycle                 +  2.71s   1.69s\n"
            "        chat                                   +  2.71s   0.49s in=1791 out=30\n"
            "          chat us.amazon.nova-2-lite-v1:0      +  2.71s   0.49s\n"
            "        execute_tool get_fraud_score           +  3.20s   1.20s  !! ERROR: card-risk service did not "
            "answer within 1.2 s (upstream 504)\n"
            "      execute_event_loop_cycle                 +  4.40s   0.67s\n"
            "        chat                                   +  4.40s   0.67s in=1868 out=42\n"
            "          chat us.amazon.nova-2-lite-v1:0      +  4.40s   0.67s"),
    "1.2": ("Recorded in a test account (6/6 sessions, 2026-09-28): execute_tool get_fraud_score "
            "status ERROR 'card-risk service did not answer within 1.2 s (upstream 504)', 1.20 s, exception event "
            "TimeoutError; execute_event_loop_cycle and invoke_agent OK, POST /invocations UNSET (HTTP 200); the "
            "Runtime's Errors metric stayed 0 while the tool failed."),
}


def s01_session_is_warm(sid) -> bool:
    """True while §0's warmed session has seen only its warm-up call(s). One session = one conversation, so a re-run
    of §1 starts a new session (its first request includes a 5-8 s cold start that no span shows)."""
    return 0 < BANK_RUNTIME.sessions.get(sid, {}).get("invocations", 0) <= OBS_STACK.probe.get("attempts", 1)


def s01_adot_facts() -> str:
    """What turns ADOT on and how much tracing code agent.py has, read from bank_runtime_agent/: the source folder §0
    packaged and deployed (the entry point is read from the live runtime by the caller)."""
    reqs = [ln.strip() for ln in (BANK_AGENT_SOURCE / "requirements.txt").read_text().splitlines()
            if ln.strip() and not ln.startswith("#") and "opentelemetry" in ln]
    code = [ln for ln in (BANK_AGENT_SOURCE / "agent.py").read_text().splitlines() if not ln.lstrip().startswith("#")]
    calls = sum(len(S01_TRACING_CALLS.findall(ln)) for ln in code)
    return (f"From {BANK_AGENT_SOURCE.name}/ (the deployed source): requirements.txt {reqs} · calls that create or "
            f"export spans in agent.py: {calls}")


def s01_transaction_search() -> str:
    try:                                             # read-only: Transaction Search is an account setting we never change
        d = SESSION.client("xray").get_trace_segment_destination()
        return f"{d.get('Destination')} / {d.get('Status')}"
    except (ClientError, BotoCoreError) as e:
        return f"unknown ({type(e).__name__})"


def s01_trace_spans(rows, tid):
    return [r for r in rows if r["trace_id"] == tid]


def s01_wall(spans) -> float:
    return (max(r["end_ns"] for r in spans) - min(r["start_ns"] for r in spans)) / 1e9 if spans else 0.0


def s01_trace_table(rows, replies):
    """Session / trace / span metrics per request (slides 11-13), all computed from the spans read back."""
    toks, out = obs.tokens_by_trace(rows), []
    for i, (p, b) in enumerate(zip(S01_JAMES, replies), 1):
        spans = s01_trace_spans(rows, b.get("trace_id"))
        wall = s01_wall(spans)
        chat = sum((r["end_ns"] - r["start_ns"]) / 1e9 for r in spans if r["name"] == "chat")
        agent_span = next((r for r in spans if r["name"].startswith("invoke_agent")), {})
        out.append({"request": f"{i} · {mc.short(p['prompt'], 34)}", "trace": f"{str(b.get('trace_id'))[:8]}…",
                    "spans": len(spans), "trace s": round(wall, 2), "model (chat) share": f"{chat / wall:.0%}" if wall else "—",
                    "input tokens: sum of chat spans": toks.get(b.get("trace_id"), {}).get("input"),
                    "input tokens: agent's own usage": (b.get("usage") or {}).get("inputTokens"),
                    "invoke_agent counter": agent_span.get("in_tok"),
                    "ERROR spans": ", ".join(r["name"] for r in spans if r["status"] == "ERROR") or "—"})
    return pd.DataFrame(out)


def s01_trace_checks(all_rows, rows, replies) -> str:
    toks = obs.tokens_by_trace(rows)
    same = sum(toks.get(b.get("trace_id"), {}).get("input") == (b.get("usage") or {}).get("inputTokens") for b in replies)
    counters = [next((r["in_tok"] for r in s01_trace_spans(rows, b.get("trace_id")) if r["name"].startswith("invoke_agent")),
                     None) for b in replies]
    kinds = {k: sum(r["kind"] == k for r in rows) for k in ("SERVER", "CLIENT")}
    earlier = len({r["trace_id"] for r in all_rows} - {r["trace_id"] for r in rows})
    sessions = {r["session_id"] for r in rows}
    # The query selects spans BY session.id, so a span without it would just be missing: check the expected shape
    # instead (one ADOT server span per trace, one botocore client span per Strands chat span).
    n_traces, n_chat = len({r["trace_id"] for r in rows}), sum(r["name"] == "chat" for r in rows)
    shape_ok = len(sessions) == 1 and n_chat > 0 and kinds["SERVER"] == n_traces and kinds["CLIENT"] == n_chat
    lines = [f"{'✓' if same == len(replies) else '✗'} Tokens: the chat spans add up to the agent's own usage in "
             f"{same}/{len(replies)} requests; the invoke_agent counters ({' → '.join(f'{c:,}' for c in counters if c)}) "
             "are cumulative for the session, so never sum them.",
             f"{'✓' if shape_ok else '✗'} session.id is on the spans Strands never created too (slide 18): SERVER "
             f"spans (Starlette) {kinds['SERVER']}/{n_traces}, one per trace · CLIENT spans (botocore → Bedrock) "
             f"{kinds['CLIENT']}/{n_chat}, one per chat span · {len(rows)} spans in all.",
             f"{'✓' if getattr(all_rows, 'complete', False) else '✗'} Every trace complete in {obs.SPANS_LOG_GROUP} "
             f"after {getattr(all_rows, 'info', {}).get('seconds')} s of polling (Logs Insights, one query per poll)."]
    if earlier:
        lines.append(f"Not shown: {earlier} earlier trace(s) of this session: §0's warm-up call (1 span, POST /invocations "
                     "only; its cold start happened before the span began).")
    return "\n".join(lines)


def s01_ancestors(rows, span) -> list[dict]:
    by_id, out, cur = {r["span_id"]: r for r in rows}, [], span
    while cur.get("parent_id") in by_id:
        cur = by_id[cur["parent_id"]]
        out.append(cur)
    return out


def s01_minute_totals(metric: dict, since: float) -> tuple[float, float]:
    """Invocations and Errors in the 1-minute periods from the minute of `since` on."""
    t_min = datetime.fromtimestamp(since // 60 * 60, timezone.utc)
    pts = (metric or {}).get("points", {})
    total = lambda m: sum(v for ts, v in pts.get(m, []) if datetime.fromisoformat(ts) >= t_min)   # noqa: E731
    return total("Invocations"), total("Errors")


def s01_wait_pillars(read, since, max_wait=60) -> dict:
    """Poll the metric + the log line until the metric counts every Runtime request this kernel sent in those minutes
    (the ledger knows them) and the log line is there. Metrics lag 10-90 s in our runs. Never raises."""
    t_min = since // 60 * 60
    expected = sum(1 for r in LEDGER.rows if r.get("agent") == "bank_assistant (Runtime)" and r.get("ts", 0) >= t_min)
    last: dict = {"inv": 0, "err": 0, "lines": [], "metric": {}}

    def probe():
        metric, lines = read()
        inv, err = s01_minute_totals(metric, since)
        last.update(metric=metric, lines=lines, inv=inv, err=err)
        return inv >= expected and bool(lines)

    ok, _, secs = mc.wait_until(probe, max_wait=max_wait, poll=5, label="the Runtime metric and the agent's log line",
                                describe=lambda _v: f"Invocations {last['inv']:.0f}/{expected} counted · log line "
                                                    f"{'found' if last['lines'] else 'not yet'}")
    return {**last, "ok": bool(ok), "expected": expected, "seconds": secs,
            "minute": datetime.fromtimestamp(t_min, timezone.utc).strftime("%H:%M")}


def s01_log_text(line: str | None, sid: str) -> str:
    if not line:
        return "not in yet (re-run the cell in a minute)"
    tid = re.search(r"trace_id=([0-9a-f]{8})", line)
    body = re.sub(r"session=\S+\s*", "", line.split("] - ", 1)[-1])       # the session is known: keep the evidence
    return mc.mask_account(f"[trace_id={tid.group(1) if tid else '?'}…] {body}", ACCOUNT)


def s01_pillar_table(p: dict, err: dict, sid: str):
    metric = (f"Invocations {p['inv']:.0f} · Errors {p['err']:.0f}" + ("" if p["inv"] >= p["expected"] else
              f"  (not all in yet: {p['inv']:.0f}/{p['expected']} requests counted)"))
    return pd.DataFrame([
        {"pillar (slide 4)": "metric", "read from": f"AWS/Bedrock-AgentCore, 1-min periods from {p['minute']} UTC",
         "what it says about request 3": metric},
        {"pillar (slide 4)": "log", "read from": "runtime log group, stream [runtime-logs-<session>]",
         "what it says about request 3": s01_log_text(p["lines"][0] if p["lines"] else None, sid)},
        {"pillar (slide 4)": "trace", "read from": f"{obs.SPANS_LOG_GROUP} via Logs Insights",
         "what it says about request 3": f"{err['name']} · {err['status']} after "
                                          f"{(err['end_ns'] - err['start_ns']) / 1e9:.2f} s: {err['status_message']}"}])


def s01_error_record(doc: dict, err: dict, rows, reply: dict) -> str:
    """The failed span as a record (slide 24), from the masked CloudWatch document, plus the success checks."""
    a, ev = doc.get("attributes", {}), next((e for e in doc.get("events", []) if e.get("name") == "exception"), {})
    ea = ev.get("attributes", {})
    parents = s01_ancestors(rows, err)
    answer = reply.get("answer") or ""
    named, stated = bool(S01_UNAVAILABLE.search(answer)), bool(S01_SCORE_STATED.search(answer))
    return "\n".join([
        "The failed span as a record (slide 24; masked, the access-key attribute dropped):",
        f"  traceId {str(doc.get('traceId'))[:8]}… (32 hex) · spanId {str(doc.get('spanId'))[:8]}… (16 hex) · "
        f"session.id …{str(a.get('session.id'))[-8:]} · {doc.get('name')} · kind {doc.get('kind')} · "
        f"{int(doc.get('durationNano', 0)) / 1e9:.2f} s",
        f"  status {doc.get('status', {}).get('code')}: {doc.get('status', {}).get('message')} · gen_ai.tool.status="
        f"{a.get('gen_ai.tool.status')} · aws.genai.span_kind={a.get('aws.genai.span_kind')}",
        f"  event '{ev.get('name')}': {ea.get('exception.type')} · stack trace "
        f"{len(str(ea.get('exception.stacktrace', '')).splitlines())} lines",
        f"{'✓' if parents and all(r['status'] != 'ERROR' for r in parents) else '✗'} Above it: "
        + " · ".join(f"{r['name']} {r['status']}" for r in parents) + " → no ERROR above the tool span",
        f"{'✓' if not reply.get('error') and answer else '✗'} The Runtime returned an answer (HTTP 200) with "
        f"tool_errors={reply.get('tool_errors')} · {'✓' if named else '✗'} it names the unavailable check · "
        f"{'✓' if not stated else '✗'} it states {'a' if stated else 'no'} fraud score"])


def s01_pillar_verdict(p: dict, err: dict, reply: dict) -> str:
    if p["inv"] < p["expected"]:
        return (f"The metric has counted {p['inv']:.0f} of this kernel's {p['expected']} requests so far (it lags "
                "10-90 s): re-run this cell for the Errors line.")
    answer = reply.get("answer") or ""
    graceful = bool(S01_UNAVAILABLE.search(answer)) and not S01_SCORE_STATED.search(answer)
    return (f"Metric: Errors {p['err']:.0f} over {p['inv']:.0f} invocations, so a dashboard says all is well. Log: "
            f"tool_errors={reply.get('tool_errors')} says why. Trace: {err['name']} says where. The request "
            f"{'still answered gracefully' if graceful else 'answered, but check the wording above'}.")


def s01_cloudwatch_added(mem, since) -> list[str]:
    """Attribute keys CloudWatch added at ingestion to one local chat span (absent in the in-memory copy)."""
    chat = next((r for r in mem if r["name"] == "chat"), None)
    if chat is None:
        return []
    docs = obs.span_documents(chat["trace_id"], session=SESSION, since=since)
    doc = next((d for d in docs if d.get("spanId") == chat["span_id"]), {})
    return sorted(set(doc.get("attributes", {})) - set(chat.get("attributes", {})))


def s01_places_table(mem, rx, cw, since):
    """In memory · OTLP receiver · CloudWatch: the same span ids? (like with like: one session, one export)."""
    ids, st = {r["span_id"] for r in mem}, LOCAL_OTEL.status

    def same(rows):
        got = {r["span_id"] for r in rows}
        return f"{'✓' if got == ids and ids else '✗'} {len(got & ids)}/{len(ids)}"
    out = [{"destination": "1 · in memory (this kernel)", "exporter": "SimpleSpanProcessor → InMemorySpanExporter",
            "spans": len(mem), "same span ids": "—", "measured": f"{len({r['trace_id'] for r in mem})} traces"}]
    if LOCAL_OTEL.receiver_ok:
        s = OTLP_RECEIVER.stats(since=since)
        out.append({"destination": "2 · OTLP receiver in this process (the enterprise backend)",
                    "exporter": "BatchSpanProcessor → OTLP/HTTP protobuf, gzip → "
                                + re.sub(r":\d+/", ":<port>/", OTLP_RECEIVER.endpoint),
                    "spans": len(rx), "same span ids": same(rx),
                    "measured": f"gzip {s['gzip_saved_pct']:.0f} % smaller ({s['raw_bytes']:,} → {s['wire_bytes']:,} "
                                f"bytes) · sender {s['user_agent']}"})
    else:
        out.append({"destination": "2 · OTLP receiver", "exporter": "—", "spans": "—", "same span ids": "—",
                    "measured": f"⚠ {st.get('receiver')}"})
    if LOCAL_OTEL.cloudwatch_ok:
        codes = sorted({e["status"] for e in LOCAL_OTEL.xray_exports if e["t"] >= since})
        out.append({"destination": f"3 · CloudWatch {obs.SPANS_LOG_GROUP} (Transaction Search)",
                    "exporter": "BatchSpanProcessor → OTLP/HTTP gzip → X-Ray OTLP endpoint, SigV4",
                    "spans": len(cw), "same span ids": same(cw),
                    "measured": f"export HTTP {codes} · read back with Logs Insights: {cw.info.get('queries')} "
                                f"quer{'y' if cw.info.get('queries') == 1 else 'ies'}, {cw.info.get('seconds')} s"})
    else:
        out.append({"destination": "3 · CloudWatch", "exporter": "—", "spans": "—", "same span ids": "—",
                    "measured": f"⚠ {st.get('cloudwatch')}"})
    return pd.DataFrame(out)


def s01_cloudwatch_note(mem, since) -> None:
    """Print the attributes CloudWatch added at ingestion (slide 25); nothing when the CloudWatch exporter is off."""
    if not LOCAL_OTEL.cloudwatch_ok:
        return
    added = s01_cloudwatch_added(mem, since)
    print(f"CloudWatch added {len(added)} attributes at ingestion that the in-memory copy of the same chat span lacks: "
          f"{', '.join(added)}" if added else "CloudWatch added no attributes to the chat span we compared.")


def s01_retry_check(mem) -> str:
    card = sorted((r for r in mem if r["name"] == "execute_tool get_card_status"), key=lambda r: r["start_ns"])
    st = [r["status"] for r in card]
    ok = st[:1] == ["ERROR"] and any(s != "ERROR" for s in st[1:])
    return (f"{'✓' if ok else '✗'} get_card_status spans in order: {' → '.join(st) or 'none'} "
            + ("(failed once, retried, succeeded)" if ok else "(no successful retry this run)"))


def s01_plot_trace(rows, tid, title, subtitle):
    viz.waterfall(obs.waterfall_rows(rows, [tid]), title=title, subtitle=subtitle)
    plt.show()


def s01_plot_runtime(rows, replies):
    if not rows or not replies or not replies[-1].get("trace_id"):
        print("⚠ No Runtime trace in this run: no waterfall (1.1 printed the recorded trace instead).")
        return
    tid = replies[-1]["trace_id"]
    spans = s01_trace_spans(rows, tid)
    err = next((r for r in spans if r["status"] == "ERROR"), None)
    what = (f"{err['name'].split()[-1]} failed after {(err['end_ns'] - err['start_ns']) / 1e9:.1f} s"
            if err else "no span failed")
    s01_plot_trace(rows, tid, f"Request 3: {what}, and the request still answered after {s01_wall(spans):.1f} s",
                   f"James's fraud check on AgentCore Runtime · trace {tid[:8]}… · read back from "
                   f"{obs.SPANS_LOG_GROUP} (Logs Insights) · ERROR spans hatched")


def s01_plot_local(mem, same_rx: bool, same_cw: bool):
    err = next((r for r in mem if r["status"] == "ERROR"), None)
    tid = err["trace_id"] if err else (mem[-1]["trace_id"] if mem else None)
    if tid is None:
        print("⚠ No local spans in memory: no waterfall.")
        return
    spans = s01_trace_spans(mem, tid)
    tries = sum(r["name"] == "execute_tool get_card_status" for r in spans)
    where = " and ".join(w for w, ok in (("the OTLP receiver", same_rx), ("CloudWatch", same_cw)) if ok)
    again = "once" if tries == 2 else f"{tries - 1} times"
    what = (f"get_card_status failed, the agent retried it {again}" if err and tries > 1
            else "get_card_status failed and was not retried" if err else "no tool failed")
    s01_plot_trace(mem, tid, f"Local agent, request 2: {what}, and answered after {s01_wall(spans):.1f} s",
                   f"Sofia's card check · spans from memory" + (f"; the same span ids reached {where}" if where else ""))


def s01_sampling_table(rows):
    """Per trace: does head 20 % keep it (decided from the trace id alone), does tail keep it (error or head)?"""
    out = []
    for tid in sorted({r["trace_id"] for r in rows}, key=lambda t: min(x["start_ns"] for x in s01_trace_spans(rows, t))):
        spans = s01_trace_spans(rows, tid)
        head, err = obs.ratio_keeps(tid, 0.2), any(r["status"] == "ERROR" for r in spans)
        out.append({"trace": f"{tid[:8]}…", "from": "Runtime (1.1)" if spans[0]["source"] == "cloudwatch" else "local (1.3)",
                    "spans": len(spans), "has an ERROR span": "yes" if err else "no",
                    "head 20 % keeps": "✓" if head else "✗", "tail (errors + 20 %) keeps": "✓" if head or err else "✗"})
    return pd.DataFrame(out)


def s01_sampling_verdict(own: dict, synth: list[dict]) -> str:
    by = {r["strategy"].split()[0]: r for r in synth}
    head, tail = by["head"], by["tail"]
    return (f"2,000 synthetic traces: head 20 % stored {head['fewer spans']} fewer spans (slide 30: 80-90 %) but kept only "
            f"{head['error traces kept']} error traces; tail kept {tail['error traces kept']} and still stored "
            f"{tail['fewer spans']} fewer spans.\n"
            f"This section's own {own['traces']} traces are too few to show a rate: by chance (random trace ids), head "
            f"20 % kept {own['head_traces']}/{own['traces']} of them and {own['head_error_traces']}/{own['error_traces']} "
            f"error traces; tail kept {own['tail_error_traces']}/{own['error_traces']} error traces, as it always does.")


def s01_recorded(part: str, why: str = "") -> None:
    print(f"⚠ {why or 'No Runtime in this run'}: showing what research recorded instead (labelled).")
    print(S01_RECORDED[part])

### 1.1 One conversation on AgentCore Runtime, read back from CloudWatch
The hosted agent's code creates no spans. **Auto-instrumentation adds spans without code changes** (slides 21, 25):
the ADOT package plus the `opentelemetry-instrument` launcher, and AgentCore injects every `OTEL_*` variable. §0
already warmed this session, so no request waits for a new microVM.

> **Deck vs 2026 (slides 21, 25):** The deck says Runtime agents get ADOT automatically. Observed in us-east-1 on
> 2026-09-28: a direct-code deployment needs `aws-opentelemetry-distro` in the zip and the entry point
> `opentelemetry-instrument agent.py`; the same zip started as `agent.py` wrote 0 spans (in our tests). The next cell prints both settings: the entry point from the live runtime, the package from the
> deployed source folder.

In [ ]:
S01_T0 = time.time()
S01_REPLIES = []
S01_SESSION_ID = None
if S01_RUNTIME_OK:
    # One session = one conversation: §0's warm session the first time, a new (cold) session when §1 is re-run.
    S01_SESSION_ID = RUNTIME_SESSION_ID if s01_session_is_warm(RUNTIME_SESSION_ID) else obs.runtime_session_id("demo", RUN_ID)
    if S01_SESSION_ID != RUNTIME_SESSION_ID:
        print("Re-run of §1: a new session (one session = one conversation), so request 1 includes a cold start.\n")
    with LEDGER.scope("§1 runtime session"):
        for _i, _payload in enumerate(S01_JAMES, 1):
            _reply = BANK_RUNTIME.invoke(_payload, S01_SESSION_ID, correlation_id=f"s01-runtime-{_i}")
            S01_REPLIES.append(_reply)
            print(f"James ({_i}): {_payload['prompt']}\n  → {_reply.get('answer') or '⚠ ' + str(_reply.get('error'))}\n"
                  f"    {_reply['wall_s']} s · trace {str(_reply.get('trace_id'))[:8]}… · tools {_reply.get('tools')} · "
                  f"tool errors {_reply.get('tool_errors')}")
    S01_CODE_CONFIG = SESSION.client("bedrock-agentcore-control").get_agent_runtime(
        agentRuntimeId=BANK_RUNTIME.runtime_id)["agentRuntimeArtifact"]["codeConfiguration"]
    print(f"\nRead from the live runtime: entryPoint {S01_CODE_CONFIG['entryPoint']}\n{s01_adot_facts()}")
print(f"CloudWatch Transaction Search (read-only): {s01_transaction_search()} → spans land in log group "
      f"{obs.SPANS_LOG_GROUP}")

In [ ]:
S01_RT_TRACES = [r["trace_id"] for r in S01_REPLIES if r.get("trace_id")]
S01_RT_ALL, S01_RT_ROWS = [], []
if S01_RT_TRACES:
    S01_RT_ALL = obs.spans_for_session(S01_SESSION_ID, session=SESSION, since=S01_T0 - 60, max_wait=45,
                                       trace_ids=S01_RT_TRACES)   # Logs Insights on aws/spans until these are complete
    S01_RT_ROWS = [r for r in S01_RT_ALL if r["trace_id"] in S01_RT_TRACES]           # this conversation's 3 traces
    print("\n".join(obs.span_tree(S01_RT_ROWS)))
    display(s01_trace_table(S01_RT_ROWS, S01_REPLIES))
    print(s01_trace_checks(S01_RT_ALL, S01_RT_ROWS, S01_REPLIES))
else:
    s01_recorded("1.1")

**What to notice.** One session, three traces, one span tree per request (9 spans each in our test runs, Sept
2026): ADOT's `POST /invocations` server span, Strands' `invoke_agent` → event-loop cycles → `chat` and
`execute_tool`, and ADOT's botocore span for each Bedrock call. Token counts come from the `chat` spans; the
Runtime's `invoke_agent` counters grow across the session.

> **Deck vs 2026 (slide 25):** The deck puts the spans in a `/aws/spans/default` log group and treats
> Transaction Search as on by default. Observed in us-east-1 on 2026-09-28: the log group is `aws/spans`, and
> Transaction Search is a one-time account setting (read above, never changed here).

<details><summary>More on where the spans went (click to open)</summary>

- The AWS docs say runtimes created since July 2026 write spans to their own log group (`spans` stream). Observed in
  us-east-1 on 2026-09-28: ours land in `aws/spans` because the course's execution role has no
  `logs:PutResourcePolicy` (in our tests).
- With indexing at about 1 %, the X-Ray trace APIs return none of these traces, so we read spans with Logs Insights.
  The CloudWatch GenAI Observability console shows the same sessions: `obs.genai_console_url()`.
- `POST /invocations` points to a parent span (AgentCore's own `InvokeAgentRuntime`, per the docs) that did not reach
  `aws/spans` in our tests (0/2 sessions); the helpers treat `POST /invocations` as the root.
- ADOT's Bedrock span carries the role's temporary access key id (`aws.auth.account.access_key`), so this notebook
  never prints raw span JSON: `obs.span_documents` drops it and masks the account id.
</details>

### 1.2 The tool failed, the request succeeded
Slide 14 on the bank agent: `get_fraud_score` timed out. **Graceful degradation means the agent keeps answering with
what still works** (balance, transactions) instead of failing the request. So alarm on failed tool spans, not only
on failed requests, and give the tool a timeout and a circuit breaker. **While the second cell waits for the
metric** (CloudWatch publishes it 10–90 s after a call), look at the waterfall.

In [ ]:
s01_plot_runtime(S01_RT_ROWS, S01_REPLIES)                  # obs.waterfall_rows -> viz.waterfall (helper collapsed above)

In [ ]:
S01_ERR = [r for r in obs.error_spans(S01_RT_ROWS) if S01_RT_TRACES and r["trace_id"] == S01_RT_TRACES[-1]]
if S01_ERR:
    S01_ERR_DOC = next((d for d in obs.span_documents(S01_ERR[0]["trace_id"], session=SESSION, since=S01_T0)
                        if d.get("spanId") == S01_ERR[0]["span_id"]), {})       # masked span record (slide 24)

    def s01_read_pillars():
        """Metric and log of the SAME request: the trace id correlates the three pillars (slide 4)."""
        metric = obs.runtime_metrics(SESSION, BANK_RUNTIME, since=S01_T0)       # AWS/Bedrock-AgentCore, 1-min
        events = SESSION.client("logs").filter_log_events(
            logGroupName=BANK_RUNTIME.log_group, startTime=int(S01_T0 * 1000) - 60_000,
            filterPattern=f'"answered" "{S01_ERR[0]["trace_id"]}"')["events"]   # the agent's own log line
        return metric, [e["message"] for e in events if "runtime-logs" in e["logStreamName"]]

    S01_PILLARS = s01_wait_pillars(s01_read_pillars, since=S01_T0)              # metrics lag 10-90 s
    display(s01_pillar_table(S01_PILLARS, S01_ERR[0], S01_SESSION_ID))
    print(s01_error_record(S01_ERR_DOC, S01_ERR[0], S01_RT_ROWS, S01_REPLIES[-1]))
    print(s01_pillar_verdict(S01_PILLARS, S01_ERR[0], S01_REPLIES[-1]))
else:
    s01_recorded("1.2", "" if not S01_RUNTIME_OK else "No ERROR span in request 3 this run")

| Slide 15 level | The question it answers | Where you saw it here |
|---|---|---|
| Session: one whole conversation | are users getting what they came for? | the tree in 1.1, one `session.id` |
| Trace: one request and its response | what made this request slow? | request 3 against request 2, in the table |
| Span: one operation | which step holds everything up? | the hatched `execute_tool` bar |

**What to notice.** The metric counts invocations that failed, and this one did not: only the log and the trace
show the failure, and the waterfall shows what the timeout cost the customer.

### 1.3 The same view for a local agent, in three places
Outside AgentCore Runtime nothing wraps your process, so instrumentation is manual: §0 built one OpenTelemetry
`TracerProvider` with three exporters (slides 21–23, 31). **OTLP is the vendor-neutral protocol of
OpenTelemetry.** A receiver inside this notebook stands in for your enterprise backend (Datadog, Splunk or an OTel
Collector would accept the same bytes), and CloudWatch takes OTLP at the X-Ray endpoint. OpenTelemetry itself stores
nothing (slide 22). Sofia asks two questions; the card platform fails once.

In [ ]:
S01_LOCAL_SID = obs.runtime_session_id("local", RUN_ID)   # Strands copies trace_attributes onto EVERY span
S01_CARD_OUTAGE = {"failures_left": 1}                     # the card platform fails once, then recovers


@tool(name="get_card_status")
def get_card_status_s01(card_last4: str) -> dict:
    """Current status (active/blocked) of a card from the card platform.

    Args:
        card_last4: Last 4 digits of the card, e.g. "0000".
    """
    if S01_CARD_OUTAGE["failures_left"]:
        S01_CARD_OUTAGE["failures_left"] -= 1
        raise ConnectionError("card platform returned HTTP 503 (temporary outage; safe to retry)")
    return {"card_last4": card_last4, "status": bank.CARD_STATUS.get(card_last4, "unknown")}


S01_LOCAL_AGENT = make_agent(
    "bank_assistant", "You are AnyCompany Bank's assistant for the signed-in customer CUST-1001 (Sofia Martinez). Use "
    "the tools for every fact. If a tool call fails, call the same tool once more with the same input before you "
    "answer; apologise only if that second call fails too. " + mc.NO_FLUFF,
    tools=[tool(bank.get_account_balances), get_card_status_s01], limits={"turns": 6},
    trace_attributes={"session.id": S01_LOCAL_SID})
S01_LOCAL_T0 = time.time()
with LEDGER.scope("§1 local agent"):
    for _i, _q in enumerate(("What's the balance in my checking account?", "Is my card ending 4417 still active?"), 1):
        print(f"Sofia ({_i}): {_q}\n  → {text_of(S01_LOCAL_AGENT(_q, invocation_state={'correlation_id': f's01-local-{_i}'}))}")
print(f"force_flush: {LOCAL_OTEL.force_flush()} s")          # the two OTLP exporters batch spans for up to 5 s

In [ ]:
S01_LOCAL_ROWS = LOCAL_OTEL.rows(S01_LOCAL_SID)                                         # 1) in memory
S01_RX_ROWS = OTLP_RECEIVER.received(S01_LOCAL_SID) if LOCAL_OTEL.receiver_ok else []   # 2) the OTLP receiver
S01_CW_ROWS = obs.spans_for_session(S01_LOCAL_SID, session=SESSION, since=S01_LOCAL_T0 - 60, min_traces=2,
                                    max_wait=30) if LOCAL_OTEL.cloudwatch_ok else []     # 3) CloudWatch aws/spans
display(s01_places_table(S01_LOCAL_ROWS, S01_RX_ROWS, S01_CW_ROWS, since=S01_LOCAL_T0))
s01_cloudwatch_note(S01_LOCAL_ROWS, S01_LOCAL_T0)
print(s01_retry_check(S01_LOCAL_ROWS))
s01_plot_local(S01_LOCAL_ROWS, *({r["span_id"] for r in x} == {r["span_id"] for r in S01_LOCAL_ROWS} != set()
                                 for x in (S01_RX_ROWS, S01_CW_ROWS)))

**What to notice.** The same span names and the same tree as on the Runtime, minus the two spans ADOT adds there
(the server span and the botocore call). This agent recovered by retrying; the Runtime agent recovered by degrading.

<details><summary>Deck vs 2026 and integration patterns (click to open)</summary>

- Slide 26 spells the variables `OTel_EXPORTER_OTLP_ENDPOINT`: OpenTelemetry ignores that spelling and falls back to
  `localhost:4318`; only uppercase `OTEL_…` works (checked in our tests). §0 sets uppercase names.
- Slides 21/25 tell you to install the ADOT SDK in agents that run outside Runtime. Observed in us-east-1 on 2026-09-28: Strands' own
  OpenTelemetry + the stock OTLP/HTTP exporter + SigV4 land complete spans in `aws/spans` with no ADOT.
- Slide 31's patterns: (1) built-in CloudWatch, (2) third-party libraries next to it (dual export: what this cell
  does), (3) an OTel Collector that fans out. The AgentCore docs say the ADOT Collector is not supported for agent
  observability into CloudWatch: use the SDK path for CloudWatch, a collector for other backends.
- Chat spans carry the whole conversation as events (prompts, tool results): decide what may leave the process.
</details>

### 1.4 Head sampling vs always-on, measured
**Head-based sampling decides when a trace starts** (by rule, slide 30), so it cannot know
whether the trace will fail. Tail-based sampling looks at the finished trace and can keep every error. First the rule
on this section's own traces, then the rates on 2,000 synthetic traces of the same shape (no model calls; nothing
leaves the notebook).

In [ ]:
S01_OWN_ROWS = S01_RT_ROWS + S01_LOCAL_ROWS                     # §1's own traces: Runtime (1.1) + local (1.3)
display(s01_sampling_table(S01_OWN_ROWS))                     # head rule: low 64 bits of the trace id < 20 % of 2^64
S01_OWN_SAMPLING = obs.head_tail(S01_OWN_ROWS, ratio=0.2)
S01_SYNTH_SAMPLING = obs.synthetic_sampling(2000, ratio=0.2, error_rate=0.1, seed=7)   # 3 private TracerProviders
display(pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in S01_SYNTH_SAMPLING]))
print(s01_sampling_verdict(S01_OWN_SAMPLING, S01_SYNTH_SAMPLING))

**What to notice.** Head sampling cuts storage roughly in proportion to its ratio, but it drops error traces at the
same rate as healthy ones. That is slide 30's trade-off: costs you can predict against keeping the traces that matter.

<details><summary>Deck vs 2026 on slide 30 (click to open)</summary>

- The deck expects 80–90 % less data from 10–20 % sampling: at 20 % the measured cut is the one printed above
  (79.6–80.8 % in our tests), the low end of that range. For gzip the deck gives 60–80 %: 1.3 printed this run's
  OTLP gzip saving (73–86 % in our tests).
- The deck's notes say tail-based sampling needs a collector. For one process it is ~30 lines
  (`obs.tail_keep_errors`); once a trace crosses several services, the decision needs a collector that sees them all.
- Always-on is what AgentCore Runtime does by default; the local provider can switch live with
  `LOCAL_OTEL.set_sampler(...)` (restore `obs.ALWAYS_ON` in a `finally`).
</details>

### Discussion prompts
<details><summary>More questions (click to open)</summary>

- Your agent answers HTTP 200 while one tool fails in 1 request out of 10. Which signal do you alarm on, and what
  would that alarm have missed in 1.2?
- Which attributes would you add to your own agents' spans (slide 9: categories, not free text), and which content
  must be redacted before spans leave the process?
- A trace crosses three agents in three services. Where does a tail-based sampler have to run?
- Which of your team's dashboards would silently double count cumulative token counters like `invoke_agent`'s?
- Try it: run 20 requests with `LOCAL_OTEL.set_sampler(TraceIdRatioBased(0.2))` and compare with 1.4 (restore
  `obs.ALWAYS_ON` in a `finally`).
</details>

## §2 — A looping agent trips two live alarms
A trace explains one request after the fact; an alarm has to catch a runaway session while it is still spending
money. *Deck slides 5, 7, 27–29.*

Slide 7 calls it a **cost blind spot**: a looping agent raises no error, every tool call succeeds, and every turn
re-reads a longer history. Slide 27 routes alerts in three tiers: **Tier 1** automated response, **Tier 2**
technical teams, **Tier 3** business stakeholders. Slide 28 lists the controls; §0 built two of them as live
10-second CloudWatch alarms (actions off, one `RunId` dimension per run):

| Slide 28 column | In this section |
|---|---|
| Proactive loop prevention | the agent's own cap, `limits={"turns": 10}` |
| Loop detection alerts | alarm: more than 5 tool calls in one session |
| Cost protection alerts | alarm: session tokens more than 2x a baseline |
| Response actions | what AgentCore Runtime offers, checked in 2.3 |

In [ ]:
LEDGER.clear_section("§2")                              # re-running §2 must not double count in §4
S02_ALARMS_UP = wait_for_alarms(max_wait=60)            # False (plus one ⚠ line) when the alarms are off or failed
S02_ALARMS_LIVE = S02_ALARMS_UP and ALARMS.wait_armed(max_wait=90)   # both OK: a new breach will show
if S02_ALARMS_UP and not S02_ALARMS_LIVE:
    print("⚠ The alarms are not back to OK since the last loop (they re-arm 70-80 s after an ALARM): 2.2 shows "
          "recorded timings.")
# The hook that publishes the metrics after every tool call (§0's); without alarms, a local one that publishes nothing.
S02_METRICS = LOOP_METRICS if S02_ALARMS_UP else obs.LoopMetrics(SESSION, RUN_ID, enabled=False)
print("The wire assistant's rule:", " ".join(x.strip() for x in re.findall(r"[^.]*PENDING[^.]*\.", obs.WIRE_PROMPT)))

In [ ]:
# Plumbing (not teaching): slide 29's alarm retyped with our own names (its curly quotes kept), the recorded
# alarm timings for a run without live alarms, a status line while a session runs, and the chart of 2.2.
from IPython.display import Pretty
from strands.hooks import AfterToolCallEvent, HookProvider

S02_SLIDE29_TEXT = """# slide 29's alarm, retyped with our own names (its straight-open, curly-close quotes kept)
cw.put_metric_alarm(
    AlarmName='SessionRunsTooLong’,
    Namespace='bedrock-agentcore’,
    MetricName='Duration’,
    Threshold=30 * 60 * 1000,   # half an hour, in ms
    Period=300,                 # five-minute windows
    Statistic='Maximum’,
    EvaluationPeriods=1,
    ComparisonOperator='GreaterThanThreshold’,
    AlarmActions=['arn:aws:sns:region:account:ops-pager']
)"""
# Recorded in a test account in us-east-1 on 2026-09-28 (the course team's runs): seconds from the datapoint that
# met the condition to ALARM. Shown, labelled, only when this run has no live alarms.
S02_RECORDED = {"loop": (13.0, 21.4, "11/11"), "cost": (16.4, 26.0, "6/6")}
S02_KIND_LABELS = {"loop": "loop detection", "cost": "cost protection"}


class ToolCallTickerS02(HookProvider):
    """One updating status line while a session runs (tool calls and tokens so far), so the loop is never a silent
    wait. It counts on its own: Strands runs AfterToolCallEvent callbacks in reverse registration order."""

    def __init__(self, label: str):
        self.label, self.calls, self.t0, self.handle = label, 0, time.time(), None

    def register_hooks(self, registry, **kwargs) -> None:
        registry.add_callback(AfterToolCallEvent, self.tick)

    def tick(self, event) -> None:
        self.calls += 1
        inv = event.agent.event_loop_metrics.latest_agent_invocation
        line = Pretty(f"{self.label}: tool call {self.calls} ({event.tool_use.get('name')}) · "
                      f"{int(inv.usage.get('totalTokens', 0)) if inv else 0:,} tokens so far · {time.time() - self.t0:.1f} s")
        if self.handle is None:
            self.handle = display(line, display_id=True)
        else:
            self.handle.update(line)


def plot_loop_alarms_s02(run: dict, cost_base: float, alarm_at: dict, breach_at: dict, live: bool, ratio: float):
    """Tokens so far after each tool call of the looping session (the values the hook published), the cost alarm's
    threshold, the moment the 10-turn cap stopped the agent, and when each alarm went to ALARM (live) or the recorded
    range after its condition was met (no live alarms). x = seconds since the looping session started."""
    t0 = run["t0"]
    xs, ys = [r["t"] - t0 for r in run["sent"]], [r["tokens"] for r in run["sent"]]
    stop = run["t_end"] - t0
    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.plot(xs, ys, color=viz.TOKEN_COLORS["input"], marker="o", markersize=5,
            label="tokens so far in the looping session (" + ("published" if run["published"] else "recorded, not "
                  "published,") + " after each tool call)")
    ax.axhline(obs.COST_MAX_RATIO * cost_base, color=viz.ALARM_COLORS["ALARM"], linestyle="--", linewidth=1.2,
               label=f"cost alarm threshold: {obs.COST_MAX_RATIO} x {cost_base:,.0f} baseline tokens")
    ax.axvline(stop, color=viz.INK_2, linewidth=2, label=f"the 10-turn cap stopped the agent ({stop:.1f} s)")
    for i, r in enumerate(run["sent"]):                  # ring + label on the datapoint that met each condition
        tags = [t for k, t in (("loop", f"loop condition (> {obs.LOOP_MAX_TOOL_CALLS} calls)"),
                               ("cost", f"cost condition (> {obs.COST_MAX_RATIO}x baseline)")) if breach_at.get(k) == r["t"]]
        if tags:
            ax.scatter(xs[i], ys[i], s=120, facecolors="none", edgecolors=viz.ALARM_COLORS["ALARM"], linewidths=1.8,
                       zorder=4)
            ax.annotate(f"call {r['call']}: " + ", ".join(tags), (xs[i], ys[i]), xytext=(stop + 0.6, ys[i] + 350),
                        textcoords="data", ha="left", va="center", fontsize=9.5, color=viz.INK_2,
                        arrowprops=dict(arrowstyle="-", color=viz.MUTED, lw=0.8, shrinkA=2, shrinkB=7))
    styles = {"loop": "-", "cost": "-."}
    ymax = max(ys + [obs.COST_MAX_RATIO * cost_base]) * 1.15
    right = stop
    for k in ("loop", "cost"):
        if live and alarm_at.get(k) is not None:
            ax.axvline(alarm_at[k], color=viz.ALARM_COLORS["ALARM"], linestyle=styles[k], linewidth=2,
                       label=f"{S02_KIND_LABELS[k]} alarm → ALARM ({alarm_at[k]:.1f} s)")
            right = max(right, alarm_at[k])
        elif not live and breach_at.get(k):              # no live alarm: the recorded range, as a labelled bar
            lo, hi, n = S02_RECORDED[k]
            b, y = breach_at[k] - t0, ymax * (0.84 if k == "loop" else 0.68)
            ax.plot([b + lo, b + hi], [y, y], color=viz.ALARM_COLORS["ALARM"], linewidth=6, alpha=0.55,
                    solid_capstyle="butt")
            ax.text(b + (lo + hi) / 2, y + ymax * 0.025, f"{S02_KIND_LABELS[k]} alarm, recorded: {lo}-{hi} s "
                    f"after its condition ({n})", ha="center", va="bottom", fontsize=9.5, color=viz.INK_2)
            right = max(right, b + hi)
    ax.set_xlim(0, right * 1.08 + 1)
    ax.set_ylim(0, ymax)
    first = min((v for v in alarm_at.values() if v is not None), default=None)
    if live and first is not None and first > stop:     # the gap the in-agent cap wins by
        y = ax.get_ylim()[1] * 0.93
        ax.annotate("", xy=(first, y), xytext=(stop, y), arrowprops=dict(arrowstyle="<->", color=viz.INK_2, lw=1.2))
        ax.text((stop + first) / 2, y, f"first ALARM {first - stop:.0f} s after the agent stopped", ha="center",
                va="bottom", fontsize=9.5, color=viz.INK_2)
    viz.thousands(ax, "y")
    ax.set_xlabel("seconds since the looping session started")
    ax.set_ylabel("tokens so far")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncols=2, fontsize=9.5)
    fired = sorted(v for v in alarm_at.values() if v is not None)
    title = f"The cap stopped the loop at {stop:.0f} s; " + (
        f"the alarms fired at {fired[0]:.0f} s and {fired[1]:.0f} s" if live and len(fired) == 2 else
        f"one alarm fired at {fired[0]:.0f} s, the other not within 60 s" if live and fired else
        "no alarm fired within 60 s" if live else
        f"recorded alarms fire {S02_RECORDED['loop'][0]:.0f}-{S02_RECORDED['cost'][1]:.0f} s after their condition")
    ax.set_title(title, pad=24)
    ax.text(0, 1.012, f"{len(run['sent'])} tool calls, {ys[-1]:,} tokens = {ratio:.1f}x the normal session · "
            + ("live alarms: 10-s periods, actions off" if live else "no live alarms in this run: recorded timings"),
            transform=ax.transAxes, fontsize=10, color=viz.INK_2, va="bottom")
    plt.show()

### 2.1 A normal session and a looping one
Sofia asks about her wire `WIRE-4471`. Both sessions use the same wire assistant: same question, prompt, model and
10-turn cap. Only the wire desk's answer differs: `COMPLETED` in the normal session, `PENDING, check again` in the loop.

In [ ]:
S02_DESKS = {"baseline": obs.WireDesk("COMPLETED"), "loop": obs.WireDesk("PENDING")}   # the only difference
S02_RUNS = {}
for _arm, _desk in S02_DESKS.items():
    _sid = obs.runtime_session_id(f"s02{_arm}", RUN_ID)                 # session.id on every span of the session
    _wire = obs.wire_agent(_desk, hooks=[S02_METRICS, ToolCallTickerS02(_arm)],      # mc.make_agent, Nova, 10 turns
                           trace_attributes={"session.id": _sid})
    _n0, _t0 = len(S02_METRICS.sent), time.time()
    with LEDGER.scope(f"§2 {_arm}"):
        _res = _wire(obs.WIRE_QUESTION, invocation_state={"correlation_id": f"s02-{_arm}"})
    S02_RUNS[_arm] = {"sid": _sid, "desk": _desk, "stop": _res.stop_reason, "reply": text_of(_res), "t0": _t0,
                      "t_end": time.time(), "n0": _n0, "sent": S02_METRICS.sent[_n0:], "published": S02_METRICS.enabled,
                      "inputs": (_wire.system_prompt, _wire.model.config["model_id"], str(_wire.default_limits))}
    print(f"{_arm}: {_res.stop_reason} after {time.time() - _t0:.1f} s", flush=True)
assert S02_RUNS["baseline"]["inputs"] == S02_RUNS["loop"]["inputs"] and "amazon.nova" in S02_RUNS["loop"]["inputs"][1]

S02_BILL = LEDGER.per_request(["§2 baseline", "§2 loop"]).set_index("request")   # tokens and $ from the ledger
S02_TABLE = pd.DataFrame([{
    "session": arm, "wire desk says": r["desk"].status, "stop reason": r["stop"], "tool calls": r["desk"].calls,
    "model calls": int(S02_BILL.loc[f"s02-{arm}", "model_calls"]),
    "tokens": int(S02_BILL.loc[f"s02-{arm}", "input_tokens"] + S02_BILL.loc[f"s02-{arm}", "output_tokens"]),
    "cost $": round(float(S02_BILL.loc[f"s02-{arm}", "cost_usd"]), 5), "wall s": round(r["t_end"] - r["t0"], 1),
    "tool spans (ERROR)": (f"{sum(s['name'].startswith('execute_tool') for s in LOCAL_OTEL.rows(r['sid']))} "
                           f"({len(obs.error_spans(LOCAL_OTEL.rows(r['sid'])))})"),
    "reply to Sofia": repr(mc.short(r["reply"], 44))} for arm, r in S02_RUNS.items()]).set_index("session")
display(S02_TABLE)
S02_RATIO = S02_TABLE.loc["loop", "tokens"] / S02_TABLE.loc["baseline", "tokens"]
S02_LOOP_SPANS = LOCAL_OTEL.rows(S02_RUNS["loop"]["sid"])
S02_LOOP_ERRORS = len(obs.error_spans(S02_LOOP_SPANS))
print(f"The loop used {S02_RATIO:.1f}x the normal session's tokens, stopped with '{S02_RUNS['loop']['stop']}' and gave "
      f"Sofia {'an empty reply' if not S02_RUNS['loop']['reply'].strip() else 'a reply'}; "
      + (f"none of its {len(S02_LOOP_SPANS)} spans is an ERROR, so an error-rate alarm would stay silent."
         if S02_LOOP_SPANS and not S02_LOOP_ERRORS else f"{S02_LOOP_ERRORS} of its {len(S02_LOOP_SPANS)} spans are ERRORs."))

### 2.2 Loop detection and cost protection, live
After every tool call, a Strands hook published two custom metrics for the session (`PutMetricData`, 1-second
resolution): `ToolCallsInSession` and `TokensInSession`. **Loop detection** means alarming on repeated tool use in
one session (slide 28 suggests more than 10 in a row; ours fires above 5). **Metric math** divides the session's
tokens by the baseline §0 measured, so the threshold reads like slide 28's "2x baseline". That is a static
threshold; slide 5's adaptive baseline would be CloudWatch anomaly detection, which has to train first.

**While it waits (~15–30 s):** compare slide 28's four columns (the table at the top of §2) with slide 27's
three tiers.

In [ ]:
S02_LOOP = S02_RUNS["loop"]
S02_COST_BASE = ALARMS.baseline_tokens if S02_ALARMS_UP else S02_TABLE.loc["baseline", "tokens"]
if S02_ALARMS_UP:                                       # the two alarms exactly as CloudWatch stores them
    for _a in SESSION.client("cloudwatch").describe_alarms(AlarmNames=list(ALARMS.names.values()))["MetricAlarms"]:
        _what = _a.get("MetricName") or next(m["Expression"] for m in _a["Metrics"] if m.get("Expression"))
        print(f"{_a['AlarmName']}: {_what} > {_a['Threshold']:g} · period "
              f"{_a.get('Period') or _a['Metrics'][0]['MetricStat']['Period']} s · actions enabled: {_a['ActionsEnabled']}")
S02_BREACH = S02_METRICS.breaches(S02_COST_BASE, since_index=S02_LOOP["n0"])   # when each condition was first met
S02_STATES, S02_ALARM_AT = {}, {"loop": None, "cost": None}
if S02_ALARMS_LIVE:
    _ok, S02_STATES, _ = ALARMS.wait_for("ALARM", since=S02_LOOP["t0"], max_wait=60)
    S02_ALARM_AT = {k: round(s["updated"] - S02_LOOP["t0"], 1) if s["state"] == "ALARM" and s["updated"] > S02_LOOP["t0"]
                    else None for k, s in S02_STATES.items()}
S02_ALARM_ROWS = pd.DataFrame([{
    "alarm": S02_KIND_LABELS[k], "condition met (s)": round(S02_BREACH[k] - S02_LOOP["t0"], 1) if S02_BREACH[k] else None,
    "agent stopped (s)": round(S02_LOOP["t_end"] - S02_LOOP["t0"], 1),
    "ALARM (s)": S02_ALARM_AT[k] if S02_ALARM_AT[k] is not None else "not within 60 s" if S02_ALARMS_LIVE else "-",
    "ALARM s after the condition": (round(S02_ALARM_AT[k] - (S02_BREACH[k] - S02_LOOP["t0"]), 1)
                                    if S02_ALARM_AT[k] is not None and S02_BREACH[k] else "-" if S02_ALARMS_LIVE
                                    else f"{S02_RECORDED[k][0]}-{S02_RECORDED[k][1]} (recorded)"),
    "reason": mc.short(S02_STATES.get(k, {}).get("reason", ""), 96)} for k in ("loop", "cost")]).set_index("alarm")
display(S02_ALARM_ROWS)
if not S02_ALARMS_LIVE:
    print("⚠ No live alarm timing in this run: the times above are recorded ones (test account, 2026-09-28, our "
          "test runs)" + ("." if S02_METRICS.enabled else "; the hook recorded the metrics without publishing them."))
_stopped = S02_LOOP["t_end"] - S02_LOOP["t0"]
_fired = {k: v for k, v in S02_ALARM_AT.items() if v is not None}
print(f"Seconds since the looping session started. The hook's last value, {S02_LOOP['sent'][-1]['tokens']:,} tokens, "
      f"{'equals' if S02_LOOP['sent'][-1]['tokens'] == S02_TABLE.loc['loop', 'tokens'] else 'differs from'} the ledger's."
      + (" Every ALARM came after the agent had stopped: " + ", ".join(
          f"{S02_KIND_LABELS[k]} {v - _stopped:.1f} s later" for k, v in _fired.items()) + "."
         if _fired and all(v > _stopped for v in _fired.values()) else ""))

In [ ]:
plot_loop_alarms_s02(S02_LOOP, S02_COST_BASE, S02_ALARM_AT, S02_BREACH, S02_ALARMS_LIVE, S02_RATIO)

**What to notice.** The in-agent cap is the brake: it stopped the loop before either alarm fired. A 10-second alarm
is the tripwire for what outlives a cap (a slow tool, a generous cap, a session on Runtime that keeps getting
requests). In our test runs (Sept 2026) each alarm fired 11–28 s after its condition was met, 8–22 s after the
agent had stopped.

<details><summary>Why it takes 10–30 seconds, and what it costs (click to open)</summary>

- Each alarm evaluates on its own fixed 10-second phase; ALARM lands about 12 s (a plain metric) or 16 s (metric
  math) after the breaching 10-second period closes (in our tests).
- After an ALARM the alarm returns to OK 70–80 s later, so a second loop within that window shows no new transition.
- Cost: a high-resolution alarm is \$0.30 per alarm-month, a custom metric \$0.30 per metric-month, `PutMetricData`
  \$0.01 per 1,000 requests (one per tool call; 69–308 ms each in our tests). §4 books them for this run. Custom
  metrics can't be deleted; 1-second data ages out after 3 hours.
- Anomaly detection (slide 5's adaptive baseline, slide 29's tip) accepted a 10-second alarm, but its model stayed
  `PENDING_TRAINING` for 30+ minutes and the alarm stayed OK through four loops in our tests: too slow to show live.
</details>

### 2.3 The deck's slide-29 alarm, checked
Slide 29 wants an alarm when a session runs longer than 30 minutes. We retype its code with our own names, run it
through Python, look for its metric, and read what AgentCore Runtime offers for a runaway session instead.

> **Deck vs 2026:** The deck alarms on `Duration` in namespace `bedrock-agentcore`; observed in us-east-1 on
> 2026-09-28, that namespace holds no `Duration` metric, so the alarm would never leave `INSUFFICIENT_DATA`.
> `AWS/Bedrock-AgentCore` has `Duration` per request, in milliseconds (the same as `Latency`), and no vended metric
> measures a whole session. The code also has curly closing quotes (a `SyntaxError`) and a placeholder SNS ARN.

In [ ]:
print(S02_SLIDE29_TEXT)
_region = re.search(r"arn:aws:sns:([^:]+):", S02_SLIDE29_TEXT).group(1)
print(f"-> AlarmActions: '{_region}' is {'a' if _region in SESSION.get_available_regions('sns') else 'not a'} Region")
try:
    compile(S02_SLIDE29_TEXT, "slide 29", "exec")
except SyntaxError as _e:
    print(f"-> SyntaxError: {_e.msg} (line {_e.lineno}: {(_e.text or '').strip()})")
S02_RUNTIME_OK = wait_for_runtime(max_wait=30)
S02_RT = BANK_RUNTIME if S02_RUNTIME_OK else None
print("Metric series:", obs.slide29_check(SESSION, S02_RT))                # read-only list_metrics
if S02_RT is not None:
    S02_FIXED = obs.slide29_corrected(S02_RT)                            # the alarm shape that has data
    _since = datetime.fromtimestamp(time.time() - 3 * 3600, timezone.utc)
    for _m in ("Duration", "Latency"):
        _pts = SESSION.client("cloudwatch").get_metric_statistics(
            Namespace=S02_FIXED["Namespace"], MetricName=_m, Dimensions=S02_FIXED["Dimensions"], StartTime=_since,
            EndTime=datetime.now(timezone.utc), Period=3 * 3600, Statistics=["Maximum", "SampleCount"])["Datapoints"]
        print(f"  {S02_FIXED['Namespace']} {_m} for our runtime, last 3 h: " + (
            f"max {max(p['Maximum'] for p in _pts):,.0f} ms over {sum(p['SampleCount'] for p in _pts):.0f} requests"
            if _pts else "no datapoints yet (they arrive 10-90 s after a request)"))
    _life = SESSION.client("bedrock-agentcore-control").get_agent_runtime(agentRuntimeId=S02_RT.runtime_id)
    print("Our runtime's session limits (lifecycleConfiguration):", _life.get("lifecycleConfiguration"))
with LEDGER.scope("§2 probe"):                          # CloudWatch API requests of this cell (ListMetrics, GetMetricStatistics)
    LEDGER.add_service("cloudwatch_api_request", 4 if S02_RT else 2, mc.AGENTCORE_PRICES["cloudwatch_api_request"],
                       correlation_id="s02-slide29", note="slide-29 check: ListMetrics + GetMetricStatistics")
_stop = SESSION.client("bedrock-agentcore").meta.service_model.operation_model("StopRuntimeSession").input_shape
print("Tier 1 response for one session: StopRuntimeSession, required:", _stop.required_members)

**What to notice.** Slide 29's question has an answer on Runtime, just not as an alarm: setting `maxLifetime=1800`
(ours has the default 28,800) would end a session's microVM 30 minutes after it starts, and `StopRuntimeSession` ends
one on demand, once something names the looping session.

<details><summary>The corrected alarm, and the limits behind slide 28 (click to open)</summary>

- Corrected slide-29 shape (per request, not per session): `obs.slide29_corrected(BANK_RUNTIME)` →
  `AWS/Bedrock-AgentCore` · `Duration` · dimensions `Name=<runtime>::DEFAULT`, `Operation=InvokeAgentRuntime`,
  `Resource=<runtime ARN>` · `Maximum` · 60 s · `TreatMissingData=notBreaching`.
- Slide 28 caps execution depth at 8 hours: 28,800 s is the default `maxLifetime` and, per the docs, the
  ceiling for microVM runtimes; the API accepts 60 s to 14 days (capacity-provider runtimes).
- To stop *the* looping session, the alarm must carry its id: a plain metric alarm doesn't. A Metrics Insights alarm
  with `GROUP BY SessionId` named the looping session (1/1 in research, ALARM 52.9 s after the breach; periods of
  60 s or more only), and its contributor event could drive a Lambda that calls `StopRuntimeSession`.
</details>

### Discussion prompts
<details><summary>More questions (click to open)</summary>

- Slide 28 suggests alerting on more than 10 consecutive uses of one tool; ours fires above 5 tool calls per
  session. What threshold fits your agents, and what would you measure first to choose it?
- Both alarms fired after the cap had stopped the loop. When is a 10-second alarm still worth its \$0.30 a month?
  *(Hint: slow tools, generous caps, Runtime sessions that keep receiving requests; its history is also
  the evidence for tuning the cap.)*
- Slide 27's tiers: which tier gets the loop alarm and which the cost alarm? *(Hint: slide 27 routes alerts by the
  kind of issue, not only by severity; one answer: the loop alarm is Tier 1 (stop the session) plus Tier 2, a
  cost trend over days is Tier 3.)*
- Slide 5 prefers adaptive baselines to static thresholds. What would your baseline be per task type or per model
  version, and how long would it need to learn?
</details>

## §3 — Grading the agent: AgentCore Evaluations with Nova judges
Traces and alarms tell us what the agent did; evaluations tell us whether it did a good job. *Deck slides 17, 33–48
and the scenario of slide 53.*

Agent answers are probabilistic, so exact-match tests fall short (slides 34–36). **AgentCore Evaluations** reads the
traces already in CloudWatch, asks an LLM judge one question per evaluator and returns a score with an explanation
(slides 37–38). Every judge here is **Amazon Nova 2 Lite**. We grade the savings coach on demand, locally (§3.2) and
on AgentCore Runtime (§3.3), then read what the online evaluation scored while we talked (§3.4).

In [ ]:
LEDGER.clear_section("§3")                          # re-running §3 never double counts in §4's bill
S03_EVALS_ON = wait_for_evaluators(max_wait=60)      # 3 derived + 1 custom evaluator, created in §0.4
S03_RUNTIME_ON = bool(S03_EVALS_ON and wait_for_runtime(max_wait=60) and wait_for_warmup(max_wait=60))
S03_COACH_RT = next((w for w in WARMUP_SESSIONS if w["persona"] == "coach"), None) if S03_RUNTIME_ON else None
print("Live parts:", f"evaluators {'on' if S03_EVALS_ON else 'OFF'} · Runtime session + warm-ups "
      f"{'on' if S03_COACH_RT else 'OFF'}")

In [ ]:
# Plumbing (not teaching): the hosted agent's module (its exact prompts and tools, loaded by path), the deck's 13
# built-ins, the labels recorded in research for the fallback, and the table / chart helpers of §3.2–§3.4.
import importlib.util
from concurrent.futures import ThreadPoolExecutor

_spec = importlib.util.spec_from_file_location("m04_bank_runtime_agent", BANK_AGENT_SOURCE / "agent.py")
S03_HOSTED = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(S03_HOSTED)
assert S03_HOSTED.DISCLAIMER == obs.DISCLAIMER, "the hosted agent's rule and the judge must name the same sentence"
S03_TURN_NAMES = ("balance", "S&P 500 fund")        # WARMUP_QUESTIONS, in order (§0.4)
assert len(WARMUP_QUESTIONS) == 2 and "S&P 500" in WARMUP_QUESTIONS[1]
S03_PERSONA_LABELS = {"coach": "coach, no rule", "coach_rule": "coach + rule"}
S03_DECK_BUILTINS = {f"Builtin.{n}" for n in (          # slide 39: 1 session + 10 trace + 2 span evaluators
    "GoalSuccessRate", "Correctness", "Faithfulness", "Helpfulness", "ResponseRelevance", "Conciseness", "Coherence",
    "InstructionFollowing", "Refusal", "Harmfulness", "Stereotyping", "ToolSelectionAccuracy", "ToolParameterAccuracy")}
S03_RECORDED_SOURCE = "recorded in a test account on 2026-09-28"
S03_RECORDED_LOCAL = {("no rule", 0): "Compliant", ("no rule", 1): "NonCompliant",   # recorded: 3/3 runs per arm,
                      ("rule", 0): "Compliant", ("rule", 1): "Compliant"}              # balance turn 12/12 Compliant
S03_GLYPH = {"pass": "✓", "partial": "~", "fail": "✗", "error": "⚠"}
S03_LEGEND = {"pass": "✓ passes the check", "partial": "~ in between", "fail": "✗ fails the check",
              "error": "⚠ error"}
S03_EVAL_NAMES = {k: (v["base"].removeprefix("Builtin.") + " (derived)" if v["kind"] == "derived"
                      else "disclaimer judge (custom)") for k, v in obs.EVALUATOR_SPECS.items()}
S03_LEVEL_ORDER = {"SESSION": 0, "TRACE": 1, "TOOL_CALL": 2}


def s03_show_judge_prompt():
    """The custom judge's instructions and scale, collapsed (click to open)."""
    import html as _html
    scale = " · ".join(f"<b>{_html.escape(s['label'])}</b>: {_html.escape(s['definition'])}"
                       for s in obs.DISCLAIMER_SCALE["categorical"])
    display(HTML(f"<details><summary>The disclaimer judge: its instructions (placeholders <code>{{context}}</code>, "
                 f"<code>{{assistant_turn}}</code>) and its categorical scale — click to open</summary>"
                 f"<pre style='white-space:pre-wrap;font-size:12px'>{_html.escape(obs.DISCLAIMER_INSTRUCTIONS)}</pre>"
                 f"<p style='font-size:12px'>{scale}</p></details>"))


def s03_cell(r):
    """(text, state) for one evaluation result row: the label and value exactly as the service returned them."""
    if not r:
        return "", None
    state = r.get("state") or obs.eval_state(r.get("label"), r.get("value"), r.get("error"))
    if r.get("error"):
        return f"⚠ {str(r['error']).split(':')[0][:24]}", "error"
    val = r.get("value")
    num = f" {val:.2f}" if isinstance(val, (int, float)) and r.get("label") not in ("Yes", "No") else ""
    return f"{S03_GLYPH[state]} {r.get('label')}{num}", state


def s03_turn(trace_id, turns):
    """Index of the turn whose trace id matches (the agent returns its request's trace id), else None."""
    return next((i for i, t in enumerate(turns) if t.get("trace_id") and t.get("trace_id") == trace_id), None)


def s03_local_table(local, judged):
    """One row per (arm, question): the reply, the keyword check (code) and the judge (live or recorded)."""
    out = []
    for arm, rec in local.items():
        by_trace = {r.get("trace_id"): r for r in judged.get(arm, [])}
        call_error = next((r for r in judged.get(arm, []) if r.get("error") and not r.get("trace_id")), None)
        for i, answer in enumerate(rec["answers"]):
            tid = rec["trace_ids"][i] if i < len(rec["trace_ids"]) else None
            r = (by_trace.get(tid) or call_error) if judged.get(arm) else None
            if not judged.get(arm):                    # evaluators off: the labels recorded in research, labelled
                r = {"label": S03_RECORDED_LOCAL[(arm, i)], "value": None, "error": None, "explanation": "",
                     "source": "recorded"}
            out.append({"arm": arm, "question": S03_TURN_NAMES[i], "reply": answer,
                        "sentence in reply (code)": obs.has_disclaimer(answer),
                        "judge": (r or {}).get("label") or (r or {}).get("error") or "no result",
                        "_row": r or {"error": "no result for this trace"},
                        "source": (r or {}).get("source", "live")})
    return pd.DataFrame(out)


def s03_local_verdict(table):
    """The measured result of §3.2, from the table (never from the model's prose)."""
    inv, bal = table[table["question"] == S03_TURN_NAMES[1]], table[table["question"] == S03_TURN_NAMES[0]]
    lab = {r["arm"]: r["judge"] for _, r in inv.iterrows()}
    code = {r["arm"]: "present" if r["sentence in reply (code)"] else "missing" for _, r in inv.iterrows()}
    if (table["source"] == "recorded").any():
        return (f"S&P 500 question — code check on today's replies: sentence {code.get('no rule')} without the rule, "
                f"{code.get('rule')} with it · judge labels {S03_RECORDED_SOURCE}: {lab.get('no rule')} without the "
                f"rule, {lab.get('rule')} with it (3/3 runs each)")
    agree = sum((r["judge"] == "Compliant") == bool(r["sentence in reply (code)"]) for _, r in inv.iterrows())
    return (f"S&P 500 question — judge: {lab.get('no rule')} without the rule, {lab.get('rule')} with it · the code "
            f"check agrees on {agree}/{len(inv)}\nBalance question — judge Compliant on "
            f"{int((bal['judge'] == 'Compliant').sum())}/{len(bal)} · the sentence is in "
            f"{int(bal['sentence in reply (code)'].sum())}/{len(bal)} replies")


def s03_local_chart(table):
    """Cards: the S&P 500 question and both replies; then a check-by-check grid (keyword check vs judge)."""
    inv = table[table["question"] == S03_TURN_NAMES[1]]
    viz.chat("Sofia Martinez (CUST-1001)", WARMUP_QUESTIONS[1], role="customer")
    for _, r in inv.iterrows():
        viz.chat(f"Savings coach, {r['arm']} (local, Nova 2 Lite)", r["reply"] or "(no answer)", role="agent")
    rows = [f"{r['arm']} · {r['question']}" for _, r in table.iterrows()]
    texts, keys = [], []
    for _, r in table.iterrows():
        code = ("✓ present", "pass") if r["sentence in reply (code)"] else ("✗ missing", "fail")
        judge = s03_cell(r["_row"])
        if r["source"] == "recorded":
            judge = (judge[0] + " (rec.)", judge[1])
        texts.append([code[0], judge[0]])
        keys.append([code[1], judge[1]])
    lab = dict(zip(inv["arm"], inv["judge"]))
    agree = sum((r["judge"] == "Compliant") == bool(r["sentence in reply (code)"]) for _, r in inv.iterrows())
    recorded = (table["source"] == "recorded").any()
    title = ("Today's replies, code check vs the judge labels recorded in a test account" if recorded else
             f"S&P 500 question: {lab.get('no rule')} without the rule, {lab.get('rule')} with it")
    sub = ("No evaluators in this run: the judge column is not live" if recorded else
           f"The code check agrees with the judge on {agree}/{len(inv)} investment replies; on the balance replies it "
           "only sees the missing sentence")
    viz.grid(rows, ["code: exact sentence in the reply", "LLM judge, recorded" if recorded else "LLM judge: Nova 2 Lite"],
             texts, keys, colors=viz.EVAL_LABEL_COLORS, legend=S03_LEGEND, figsize=(9.5, 4.2), wrap=24, title=title,
             subtitle=sub)
    plt.show()


def s03_runtime_table(rows, w):
    """§3.3's results, session -> trace -> tool call, with what each result scored."""
    out = []
    for r in rows:
        i = s03_turn(r.get("trace_id"), w["turns"])
        lvl = r.get("level") or "?"
        if lvl == "SESSION":
            scored = "the whole session"
        elif lvl == "TRACE":
            scored = f"turn {i + 1}: {S03_TURN_NAMES[i]}" if i is not None else f"trace …{str(r.get('trace_id'))[-6:]}"
        else:
            used = (w["turns"][i].get("tools") or []) if i is not None else []
            scored = (f"turn {i + 1}: tool call {used[0]}" if len(used) == 1 else
                      f"turn {i + 1}: a tool call" if i is not None else f"tool span …{str(r.get('span_id'))[-6:]}")
        text, _state = s03_cell(r)
        out.append({"level": lvl, "evaluator": S03_EVAL_NAMES.get(r["evaluator"], r["evaluator"]), "scored": scored,
                    "result": text, "judge's explanation": mc.short(r.get("explanation") or r.get("error") or "", 110)})
    return pd.DataFrame(sorted(out, key=lambda x: (S03_LEVEL_ORDER.get(x["level"], 9), x["scored"])))


def s03_eval_cost_line():
    """What §3's on-demand evaluations cost so far, from the ledger: per-evaluation fees + the Nova judge's tokens."""
    d = LEDGER.df(["§3 evaluate local", "§3 evaluate runtime"])
    if d.empty:
        return "No on-demand evaluations booked yet."
    fees, judge = d[d["model"] == "AgentCore"], d[d["model"] != "AgentCore"]
    return (f"§3 on demand so far (ledger): {int(fees['service_units'].sum())} evaluations · "
            f"${fees['cost_usd'].sum():.4f} in fees ($0.0015 each) + ${judge['cost_usd'].sum():.4f} of Nova judge "
            f"tokens ({int(judge['input_tokens'].sum()):,} in / {int(judge['output_tokens'].sum()):,} out)")


def s03_config_table(desc):
    """The online evaluation config as the service describes it (masked), evaluator ids shown as our keys."""
    keys = {v: k for k, v in EVALUATORS.ids.items()}
    rows = {"status": f"{desc.get('status')} · {desc.get('execution')}",
            "sampling": f"{desc.get('sampling %')} %",
            "session ends after": f"{desc.get('session idle timeout (min)')} idle minute(s)",
            "data source: log group": ", ".join(mc.middle(str(x), 44, 22) for x in desc.get("log groups") or []),
            "data source: service name": ", ".join(desc.get("service names") or []),
            "evaluators": ", ".join(S03_EVAL_NAMES.get(keys.get(e), e) for e in desc.get("evaluators") or []),
            "results go to": mc.middle(str(desc.get("results log group")), 44, 22)}
    return pd.DataFrame([{"online evaluation config": k, "value": v} for k, v in rows.items()])


def s03_dedupe(rows):
    """One row per (evaluator, session, trace): the latest if the service wrote a result twice."""
    last = {}
    for r in sorted(rows, key=lambda r: str(r.get("logged_at") or "")):
        last[(r.get("evaluator"), r.get("session_id"), r.get("trace_id") if r.get("level") != "Session" else "")] = r
    return list(last.values())


def s03_minutes_after(rows, t_end):
    """Minutes from the sessions' last answer to the time the results were logged (online results only)."""
    ts = []
    for r in rows:
        try:
            ts.append(datetime.strptime(str(r["logged_at"])[:23], "%Y-%m-%d %H:%M:%S.%f")
                      .replace(tzinfo=timezone.utc).timestamp())
        except (KeyError, ValueError):
            continue
    return (max(ts) - t_end) / 60 if ts else None


def s03_agreement(on_demand, scored, sid):
    """Same session, same evaluators: on demand (§3.3) vs online/batch (§3.4) -> (same, compared, differences)."""
    def key(r):
        return (r.get("evaluator"), "session" if str(r.get("level")).upper() == "SESSION" else r.get("trace_id"))
    a = {key(r): r.get("label") for r in on_demand if r.get("session_id") in (sid, None) and not r.get("error")}
    b = {key(r): r.get("label") for r in scored if r.get("session_id") == sid and not r.get("error")}
    both = sorted(set(a) & set(b), key=str)
    diffs = [f"{k[0]}: {a[k]} vs {b[k]}" for k in both if a[k] != b[k]]
    return len(both) - len(diffs), len(both), diffs


def s03_scored_chart(rows, source, seconds=None):
    """The warm-up conversations as scored online (or by the batch fallback): one row per turn, one column per
    evaluator; the session-level result sits on the session's first row."""
    cols = [("helpful", "Helpfulness (trace)"), ("disclaimer", "disclaimer judge (trace)"),
            ("goal", "GoalSuccessRate (session)")]
    present = {r["evaluator"] for r in rows}
    if source == "batch":                              # the fallback runs only the disclaimer judge + GoalSuccessRate
        cols = [c for c in cols if c[0] in present]
    labels, texts, keys = [], [], []
    for w in sorted(WARMUP_SESSIONS, key=lambda w: w["persona"]):
        mine = [r for r in rows if r.get("session_id") == w["session_id"]]
        for i, t in enumerate(w["turns"]):
            labels.append(f"{S03_PERSONA_LABELS.get(w['persona'], w['persona'])} · {S03_TURN_NAMES[i]}")
            trow, krow = [], []
            for k, _ in cols:
                if k not in present:
                    cell = ("no result yet", None)
                elif k == "goal":
                    cell = s03_cell(next((r for r in mine if r["evaluator"] == k), None)) if i == 0 else ("↑ per session", None)
                else:
                    cell = s03_cell(next((r for r in mine if r["evaluator"] == k and r.get("trace_id") == t.get("trace_id")),
                                         None))
                    cell = cell if cell[1] else ("no result yet", None)
                trow.append(cell[0])
                krow.append(cell[1])
            texts.append(trow)
            keys.append(krow)
    ended = max(w["t_end"] for w in WARMUP_SESSIONS)
    if source == "online":
        lag = s03_minutes_after(rows, ended)
        title = f"Online evaluation: {len(rows)} results, logged {lag:.0f} min after the warm-ups ended" if lag is not None \
            else f"Online evaluation: {len(rows)} results for the warm-up conversations"
    else:
        title = f"Batch evaluation (fallback): {len(rows)} results in {seconds:.0f} s"
    same, n, _d = s03_agreement(S03_RT_ROWS, rows, S03_COACH_RT["session_id"]) if S03_COACH_RT else (0, 0, [])
    sub = (f"The same session on demand (§3.3) got the same label on {same}/{n} results" if n
           else "Rows: the two warm-up conversations §0.4 sent to the hosted coach")
    viz.grid(labels, [c for _, c in cols], texts, keys, colors=viz.EVAL_LABEL_COLORS, legend=S03_LEGEND,
             figsize=(9.5, 4.2), wrap=20, title=title, subtitle=sub)
    plt.show()

### 3.1 Built-in, derived and custom evaluators
An **evaluator** scores one level: a session, a trace (one request) or a span (one tool call); the API says `SESSION`,
`TRACE`, `TOOL_CALL` (slide 39). **Built-in** evaluators (`Builtin.<Name>`) keep their prompt hidden and cannot be
modified (slide 41). A **derived evaluator** re-uses a built-in's prompt and scale on your judge model: our
Helpfulness, GoalSuccessRate and ToolSelectionAccuracy run on Nova 2 Lite. A **custom evaluator** carries your own
prompt, judge model, scale and level (slide 42): our disclaimer judge.

> **Deck vs 2026:** The deck says 13 built-in evaluators (slide 39), usable only as templates (slide 41). Observed in
> us-east-1 on 2026-09-28: more built-ins plus third-party ones (the cell prints today's catalog), and the derived
> evaluator is the API for "template".

In [ ]:
S03_CATALOG = obs.catalog(SESSION)                   # ListEvaluators: every AWS-managed evaluator in the Region
if "error" in S03_CATALOG:
    print(f"⚠ ListEvaluators failed ({S03_CATALOG['error']}): the catalog count is skipped.")
else:
    print(f"AWS built-ins today: {S03_CATALOG['builtin']} (" + ", ".join(
        f"{k.split()[1]} {v}" for k, v in S03_CATALOG["by_type_level"].items() if k.startswith("Builtin"))
        + f") + {S03_CATALOG['third_party']} third-party · the deck (slide 39): {len(S03_DECK_BUILTINS)} built-ins")
    print("Built-ins the deck does not list:", ", ".join(
        i.removeprefix("Builtin.") for i in sorted(set(S03_CATALOG["builtin_ids"]) - S03_DECK_BUILTINS)) or "none")
print("The deck's package agentcore_evaluation (slides 43, 45, 47) installed:",
      importlib.util.find_spec("agentcore_evaluation") is not None)
if S03_EVALS_ON:
    S03_CONFIG = pd.DataFrame(EVALUATORS.config_rows())      # GetEvaluator on this run's four evaluators
    display(S03_CONFIG[["key", "level", "type", "base", "judge model", "scale"]])
    print(f"Judge = Amazon Nova on {sum('amazon.nova' in str(m) for m in S03_CONFIG['judge model'])}/{len(S03_CONFIG)} "
          f"evaluators · AWS-managed built-ins called as-is: {int((S03_CONFIG['type'] == 'Builtin').sum())}")
    s03_show_judge_prompt()

### 3.2 On demand: the savings coach, with and without the rule
The deck's scenario (slide 53): compliance needs every investment reply to carry a disclaimer, and no built-in covers
it. Our custom judge applies the bank's rule; next to it, code checks whether the exact sentence `obs.DISCLAIMER` is in
the reply. **On-demand evaluation** is synchronous and developer-initiated: `Evaluate` gets a session's spans and
returns the label in the same call (slides 40, 46). Here the spans come from memory, so nothing waits for CloudWatch.

Two local coaches run the hosted agent's exact prompts, without and with the rule, on §0's two warm-up questions.
**Watch for:** only the S&P 500 question needs the disclaimer.

In [ ]:
S03_ARMS = {"no rule": "coach", "rule": "coach_rule"}   # arm -> the hosted agent's persona (bank_runtime_agent/agent.py)
S03_LOCAL = {arm: {"session_id": obs.runtime_session_id(f"s03{persona}", RUN_ID), "answers": [], "trace_ids": [],
                   "error": None} for arm, persona in S03_ARMS.items()}


def s03_run_coach(arm):
    """One local savings-coach conversation: the hosted agent's prompt, tools and model; the same two questions."""
    persona, rec = S03_ARMS[arm], S03_LOCAL[arm]
    coach = make_agent(f"savings_{persona}", S03_HOSTED.PROMPTS[persona], limits={"turns": 6},
                       tools=[S03_HOSTED.get_account_balances, S03_HOSTED.list_card_transactions,
                              S03_HOSTED.get_fraud_score], trace_attributes={"session.id": rec["session_id"]})
    try:
        with LEDGER.local_scope("§3 local coach"):
            for i, q in enumerate(WARMUP_QUESTIONS):
                rec["answers"].append(text_of(coach(f"[customer_id=CUST-1001] {q}",
                                                    invocation_state={"correlation_id": f"s03-{persona}-{i}"})))
    except Exception as e:  # noqa: BLE001 — a throttled arm is reported, never a crash
        rec["error"] = mc.short(f"{type(e).__name__}: {e}", 120)


with ThreadPoolExecutor(max_workers=2) as _pool:        # both arms at once: same questions, model, tools, concurrency
    _jobs = [_pool.submit(s03_run_coach, arm) for arm in S03_ARMS]
    mc.wait_until(lambda: all(j.done() for j in _jobs), max_wait=90, poll=1, label="two local savings coaches",
                  describe=lambda _v: " · ".join(f"{a}: {len(r['answers'])}/2 replies" for a, r in S03_LOCAL.items()))
LOCAL_OTEL.force_flush()
for _arm, _rec in S03_LOCAL.items():                    # turn order = start order of the invoke_agent spans
    _rec["trace_ids"] = [r["trace_id"] for r in LOCAL_OTEL.rows(_rec["session_id"]) if r["name"].startswith("invoke_agent")]
    print(f"{_arm:>8}: {len(_rec['answers'])} replies, {len(LOCAL_OTEL.readable_spans(_rec['session_id']))} spans in "
          "memory" + (f" · ⚠ {_rec['error']}" if _rec["error"] else ""))

In [ ]:
S03_LOCAL_EVAL = {}
if S03_EVALS_ON:
    with LEDGER.scope("§3 evaluate local"):
        for _arm, _rec in S03_LOCAL.items():       # one Evaluate call per arm: the custom judge scores both traces
            S03_LOCAL_EVAL[_arm] = EVALUATORS.evaluate_local(LOCAL_OTEL.readable_spans(_rec["session_id"]),
                                                             keys=("disclaimer",), correlation_id=f"s03-eval-{_arm}")
else:
    print("⚠ No evaluators in this run: the judge column shows the labels " + S03_RECORDED_SOURCE + ".")
S03_LOCAL_TABLE = s03_local_table(S03_LOCAL, S03_LOCAL_EVAL)   # + the code check on every reply
print(s03_local_verdict(S03_LOCAL_TABLE))
_no_rule = S03_LOCAL_TABLE[(S03_LOCAL_TABLE["arm"] == "no rule") & (S03_LOCAL_TABLE["question"] == S03_TURN_NAMES[1])]
if S03_EVALS_ON and len(_no_rule):
    print("Judge's explanation (no rule, S&P 500):", mc.short(_no_rule.iloc[0]["_row"].get("explanation") or "—", 260))

In [ ]:
s03_local_chart(S03_LOCAL_TABLE)                       # the S&P 500 replies, then code check vs judge per reply

**What to notice.** The code check only sees the sentence; the judge also decides whether a reply *is* investment
guidance, so the balance replies pass without it. In our test runs (Sept 2026) the disclaimer labels here and in §3.3
were the same in every run (10/10 local, 9/9 Runtime); only the replies' wording changed.

> **Deck vs 2026:** Slides 43 and 47 import `agentcore_evaluation`; observed on 2026-09-28 no such package exists
> (§3.1 printed the check). The real calls are boto3 `create_evaluator` / `evaluate` or `bedrock_agentcore.evaluation`,
> and several slide values are rejected.

<details><summary>Slides 43 and 47, corrected (click to open)</summary>

| Slide | Deck | What the API takes (verified in a test account, 2026-09-28) |
|---|---|---|
| 43 | a hyphenated evaluator name such as `"my-judge"` | `evaluatorName` must match `[a-zA-Z][a-zA-Z0-9_]{0,47}`: hyphens are rejected |
| 43 | the level spelled `"Session"` | `level` is `SESSION`, `TRACE` or `TOOL_CALL` |
| 43 | `{context}` and `{agent_response}` | `{agent_response}` is not a placeholder; a TRACE judge reads `{assistant_turn}` |
| 43 | a scale given as a dictionary keyed by score | `ratingScale.numerical[{value, label, definition}]` or `.categorical[{label, definition}]` |
| 47 | one `evaluate` call with a session id and a list of evaluators | `Evaluate` takes ONE `evaluatorId` plus the session's span documents (and up to 10 target ids); the SDK's `EvaluationClient.run(evaluator_ids, session_id, …)` loops for you |

Checked with boto3 and `bedrock_agentcore.evaluation` in a test account in us-east-1.
</details>

### 3.3 On demand: a Runtime session from CloudWatch
The same evaluators on a conversation the hosted coach (no rule) had on AgentCore Runtime during §0.4. Its spans come
back from `aws/spans`, its prompts and replies from the runtime log group; the helper waits until every agent span has
its record, or `Evaluate` answers HTTP 200 with a per-result error. One call per evaluator scores every unit of its
level (slides 39, 46–47), and each call is a Nova `Converse` in this account: a fee plus judge tokens, printed from the
ledger. Judges vary too: in 1 of our 9 runs ToolSelectionAccuracy called the second balance lookup "No", so read the
explanation, not only the label (slide 41).

In [ ]:
S03_RT_ROWS = []
if S03_COACH_RT:
    with LEDGER.scope("§3 evaluate runtime"):
        S03_RT_ROWS = EVALUATORS.evaluate_runtime(S03_COACH_RT["session_id"], BANK_RUNTIME, S03_COACH_RT["t_start"] - 60,
                                                  keys=("goal", "helpful", "disclaimer", "toolsel"),
                                                  min_traces=len(S03_COACH_RT["turns"]), correlation_id="s03-eval-runtime")
    print(f"The hosted coach without the rule, {(time.time() - S03_COACH_RT['t_end']) / 60:.1f} min after its last "
          f"answer: {len(S03_RT_ROWS)} results from {len({r['evaluator'] for r in S03_RT_ROWS})} evaluators")
    display(s03_runtime_table(S03_RT_ROWS, S03_COACH_RT))
    print(s03_eval_cost_line())
else:
    print("⚠ No Runtime session to evaluate in this run (see the ⚠ lines at the top of §3). Recorded in a test account "
          "on 2026-09-28: the same disclaimer judge on the hosted coach's "
          "S&P 500 reply said NonCompliant 2/2 without the rule and Compliant 2/2 with it.")

### 3.4 Online: scored while we talked
**Online evaluation** scores sampled live traffic asynchronously, with no change to the agent (slides 40, 44, 48).
§0.4 attached a config to the Runtime agent's log group with Helpfulness, GoalSuccessRate and the disclaimer judge.
Every ~5 minutes the service collects sessions that went idle in a window closing 5 minutes earlier, so results land
~10 minutes after a session ends (6–11 min in our runs, Sept 2026): the warm-ups ended during §0.

**If their results are not in yet**, the cell runs a **batch evaluation** of the same two sessions (~60–70 s).

> **Deck vs 2026:** Slide 45 passes an agent ARN and a sampling rate of 0.1, meant as 10 %. Observed in us-east-1 on 2026-09-28:
> the config takes the runtime's log group plus the service name `<runtime>.DEFAULT`, and `samplingPercentage` is a
> percent (0.1 means 0.1 %).

In [ ]:
S03_SCORED, S03_SCORED_BY, S03_BATCH = [], None, {}
if S03_COACH_RT and ONLINE_EVALUATION is not None:
    S03_SIDS = [w["session_id"] for w in WARMUP_SESSIONS]
    S03_ENDED = max(w["t_end"] for w in WARMUP_SESSIONS)
    S03_EXPECTED = sum(2 * len(w["turns"]) + 1 for w in WARMUP_SESSIONS)   # helpful + disclaimer per turn, goal once
    if wait_for_online_eval(max_wait=30):
        display(s03_config_table(ONLINE_EVALUATION.describe()))
        S03_SCORED, S03_SCORED_BY = s03_dedupe(ONLINE_EVALUATION.results(session_ids=S03_SIDS)), "online"
        print(f"Warm-ups ended {(time.time() - S03_ENDED) / 60:.1f} min ago · online results for them so far: "
              f"{len(S03_SCORED)} of {S03_EXPECTED}")
    if not S03_SCORED:                              # the fallback: the same sessions, scored in ~1 minute
        S03_BATCH = ONLINE_EVALUATION.batch(BANK_RUNTIME, [EVALUATORS.ids[k] for k in ("disclaimer", "goal")],
                                            S03_SIDS, ended_at=S03_ENDED, max_wait=90)
        S03_SCORED, S03_SCORED_BY = S03_BATCH["rows"], "batch"
        print(f"Batch evaluation: {S03_BATCH['status']} after {S03_BATCH.get('seconds', 0)} s, {len(S03_SCORED)} results"
              + (f" · {S03_BATCH['error']}" if S03_BATCH.get("error") else ""))
        if S03_BATCH["status"] == "TOO_EARLY":
            print(f"⚠ The warm-ups are too young for a batch (it needs sessions ~5 min old in our test runs, "
                  f"2026-09-28). Re-run this cell in ~{S03_BATCH['wait_s'] / 60:.0f} min; online results are due "
                  f"~{max(0.0, (S03_ENDED + 600 - time.time()) / 60):.0f} min from now.")
else:
    print("⚠ No online evaluation in this run (no Runtime, warm-ups or evaluators: see the ⚠ lines at the top of §3). "
          "Recorded in a test account on 2026-09-28: online and batch "
          "results gave the same labels as on demand.")

In [ ]:
if S03_SCORED:
    s03_scored_chart(S03_SCORED, S03_SCORED_BY, S03_BATCH.get("seconds"))
    _same, _n, _diffs = s03_agreement(S03_RT_ROWS, S03_SCORED, S03_COACH_RT["session_id"])
    print(f"Coach session, on demand (§3.3) vs {S03_SCORED_BY}: same label on {_same}/{_n}"
          + (f" · differs: {'; '.join(_diffs)}" if _diffs else ""))
    for _r in [r for r in S03_SCORED if r.get("error")][:3]:
        print(f"⚠ {_r['evaluator']} on session …{str(_r.get('session_id'))[-6:]}: {_r['error']}")

**What to notice.** Same evaluators, same judge: on-demand and online labels are comparable (slide 40); only when
they arrive and who starts them differ. Scores also land as CloudWatch metrics (`Bedrock-AgentCore/Evaluations`), so a
quality drop can raise an alarm like §2's (slide 48).

### Discussion prompts
**Think about it:** Pick one rule your own agents must follow (a disclaimer, a limit, a tone). Would you check it
with code, with an LLM judge, or both, and would you run that check on demand before release, online in production,
or both?

<details><summary>More questions (click to open)</summary>

- The Helpfulness judge's explanation is often more useful than its score (slide 41). Where would you surface it for
  the team that owns the agent?
- Online evaluation costs a fee per evaluation plus judge tokens. What sampling percentage would you start with for a
  bank agent that handles 50,000 conversations a day, and what would make you raise it?
- A derived evaluator runs a hidden AWS prompt on your judge model. How would you check that a Nova judge agrees with
  your compliance team before you trust its labels (slide 42: 20–50 human-labelled examples)?
- Which failures from slide 35 would the disclaimer judge never catch, and which evaluator or alarm would?
</details>

## §4 — Wrap-up: what we measured, what it cost, cleanup
Let's put the four highlights' numbers side by side, add up what they cost, and clean up. *Slides 6, 49, 55.*

The first cell books what bills by time or volume while the resources still exist, then starts the cleanup **in the
background**. The rest of §4 reads the ledger and the numbers each section already checked.

In [ ]:
LEDGER.clear_section("§4")                                   # a re-run of §4 never books its charges twice
if globals().get("WRAPUP_CLEANUP_S04") is None:              # first run: read the charges, THEN start the cleanup
    try:                                                     # Runtime hours, online/batch evaluations, alarms, CloudWatch
        WRAPUP_SERVICE_ROWS_S04 = observability_cost_rows()
    except Exception as _e:  # noqa: BLE001 — the cleanup must start even if a charge cannot be read
        WRAPUP_SERVICE_ROWS_S04 = []
        print(f"⚠ The time- and volume-based charges could not be read ({type(_e).__name__}): 4.2 lacks them.")
    WRAPUP_STEPS_S04 = observability_cleanup_steps()         # §0's contract, in dependency order ([] on a dev stack)
    WRAPUP_CLEANUP_S04 = mc.Cleanup().start(WRAPUP_STEPS_S04)   # one background thread; each step isolated
    WRAPUP_T0_S04 = time.time()
    if WRAPUP_STEPS_S04:
        print(f"Cleanup of RUN_ID {RUN_ID} started in the background, {len(WRAPUP_STEPS_S04)} steps in this order:"
              + "".join(f"\n  {i}. {label}" for i, (label, _fn) in enumerate(WRAPUP_STEPS_S04, 1)))
else:
    print("§4 already ran in this kernel: the cleanup started then, and its charges are re-booked from that reading.")
with LEDGER.scope("§4 services"):
    for _row in WRAPUP_SERVICE_ROWS_S04:
        LEDGER.add_service(_row["service"], _row["units"], _row["unit_price"], note=_row["note"],
                           correlation_id="s04-services")
print(f"Booked {len(WRAPUP_SERVICE_ROWS_S04)} time- or volume-based charges: "
      f"${sum(r['usd'] for r in WRAPUP_SERVICE_ROWS_S04):.5f}")

In [ ]:
# Plumbing (not teaching): the "what we measured" rows, read from the values each section computed and checked in its
# own cells (a section that did not run, or ran its fallback, says so), and the cost of this run from the LEDGER.
import math


def wrapup_get_s04(name: str, default=None):
    """A value another section computed; the default when that section did not run in this kernel."""
    return globals().get(name, default)


def wrapup_runtime_s04() -> str | None:
    rows, replies = wrapup_get_s04("S01_RT_ROWS") or [], wrapup_get_s04("S01_REPLIES") or []
    if not rows:
        return None
    order = sorted({r["trace_id"] for r in rows}, key=lambda t: min(r["start_ns"] for r in rows if r["trace_id"] == t))
    per = [sum(r["trace_id"] == t for r in rows) for t in order]
    secs = getattr(wrapup_get_s04("S01_RT_ALL"), "info", {}).get("seconds")
    spans = f"{per[0]} spans each" if len(set(per)) == 1 else f"{'/'.join(map(str, per))} spans"
    return (f"{len(replies)} requests in one session: {len(order)} traces, {spans}, all complete in "
            f"{obs.SPANS_LOG_GROUP} after {secs} s")


def wrapup_failure_s04() -> str | None:
    err, replies = (wrapup_get_s04("S01_ERR") or [None])[0], wrapup_get_s04("S01_REPLIES")
    pil = wrapup_get_s04("S01_PILLARS")
    if not err or not replies:
        return None
    last, spans = replies[-1], [r for r in wrapup_get_s04("S01_RT_ROWS") or [] if r["trace_id"] == err["trace_id"]]
    trace_s = (max(r["end_ns"] for r in spans) - min(r["start_ns"] for r in spans)) / 1e9
    answered = (f"still answered: {trace_s:.2f} s end to end in its trace" if last.get("answer") and not last.get("error")
                else "returned no answer")
    metric = (f" · Runtime Errors metric {pil['err']:.0f} over {pil['inv']:.0f} invocations"
              if pil and pil.get("inv", 0) >= pil.get("expected", 1) else "")
    return (f"{err['name'].split()[-1]} ERROR after {(err['end_ns'] - err['start_ns']) / 1e9:.2f} s · the request "
            f"{answered}{metric}")


def wrapup_local_s04() -> str | None:
    mem = wrapup_get_s04("S01_LOCAL_ROWS") or []
    if not mem:
        return None
    ids = {r["span_id"] for r in mem}

    def same(name):
        got = {r["span_id"] for r in (wrapup_get_s04(name) or [])}
        return f"{len(got & ids)}/{len(ids)}" if got else "off"
    return (f"{len(ids)} spans in memory; the same span ids at the OTLP receiver {same('S01_RX_ROWS')} and in "
            f"CloudWatch {same('S01_CW_ROWS')}")


def wrapup_sampling_s04() -> str | None:
    by = {r["strategy"].split()[0]: r for r in wrapup_get_s04("S01_SYNTH_SAMPLING") or []}
    if not {"head", "tail"} <= set(by):
        return None
    h, t = by["head"], by["tail"]
    return (f"2,000 synthetic traces · head 20 %: {h['fewer spans']} fewer spans, {h['error traces kept']} error traces "
            f"kept · tail: {t['fewer spans']} fewer, {t['error traces kept']} kept")


def wrapup_loop_s04() -> str | None:
    table = wrapup_get_s04("S02_TABLE")
    if table is None or "loop" not in table.index:
        return None
    loop, base = table.loc["loop"], table.loc["baseline"]
    return (f"{loop['tokens']:,} vs {base['tokens']:,} tokens ({wrapup_get_s04('S02_RATIO'):.1f}x), "
            f"{loop['tool calls']} tool calls, stop reason {loop['stop reason']} · "
            f"{wrapup_get_s04('S02_LOOP_ERRORS')} ERROR spans of {len(wrapup_get_s04('S02_LOOP_SPANS') or [])}")


def wrapup_alarms_s04() -> str | None:
    loop, at = wrapup_get_s04("S02_LOOP"), wrapup_get_s04("S02_ALARM_AT") or {}
    if not loop:
        return None
    stop = f"the 10-turn cap stopped the agent at {loop['t_end'] - loop['t0']:.1f} s"
    if not wrapup_get_s04("S02_ALARMS_LIVE"):
        return stop + " · no live alarms in this run (§2.2 showed recorded timings)"
    fired = " · ".join(f"{k} ALARM at {v:.1f} s" if v is not None else f"{k}: no ALARM within 60 s"
                       for k, v in at.items())
    return f"{stop} · {fired} (seconds after the loop started)"


def wrapup_on_demand_s04() -> str | None:
    table, turns = wrapup_get_s04("S03_LOCAL_TABLE"), wrapup_get_s04("S03_TURN_NAMES")
    if table is None or not turns:
        return None
    inv = table[table["question"] == turns[1]]
    lab = dict(zip(inv["arm"], inv["judge"]))
    agree = sum((r["judge"] == "Compliant") == bool(r["sentence in reply (code)"]) for _, r in inv.iterrows())
    text = (f"local {turns[1]} reply: {lab.get('no rule')} without the rule, {lab.get('rule')} with it; the code check "
            f"agrees {agree}/{len(inv)}" + (" (judge labels recorded)" if (table["source"] == "recorded").any() else ""))
    rt, coach = wrapup_get_s04("S03_RT_ROWS") or [], wrapup_get_s04("S03_COACH_RT")
    if rt and coach:
        tid = coach["turns"][1].get("trace_id")
        disc = next((r.get("label") for r in rt if r["evaluator"] == "disclaimer" and r.get("trace_id") == tid), None)
        text += (f" · Runtime session: {sum(not r.get('error') for r in rt)}/{len(rt)} results, disclaimer on the "
                 f"{turns[1]} turn: {disc}")
    return text


def wrapup_online_s04() -> str | None:
    by, scored, batch = (wrapup_get_s04("S03_SCORED_BY"), wrapup_get_s04("S03_SCORED") or [],
                         wrapup_get_s04("S03_BATCH") or {})
    rt, coach = wrapup_get_s04("S03_RT_ROWS") or [], wrapup_get_s04("S03_COACH_RT")
    agreement = wrapup_get_s04("s03_agreement")               # §3.4's own comparison function
    same = ""
    if scored and rt and coach and agreement:                # the same session, on demand (§3.3) vs scored (§3.4)
        n_same, n, _diffs = agreement(rt, scored, coach["session_id"])
        same = f" · same label as on demand {n_same}/{n}" if n else ""
    if by == "online" and scored:
        return (f"{len(scored)} online results for the two warm-up conversations, scored by the service with no call "
                f"from this notebook ({sum(bool(r.get('error')) for r in scored)} with an error){same}")
    if by == "batch" and batch.get("status") == "TOO_EARLY":
        age = (obs.OnlineEvaluation.BATCH_MIN_AGE_S - batch.get("wait_s", 0)) / 60
        return (f"no results yet: the warm-ups were {age:.1f} min old at §3.4 (online results come ~10 min after a "
                "session ends; a batch needs ~5 min)")
    if by == "batch":
        return (f"batch fallback: {batch.get('status')}, {len(scored)} results in {batch.get('seconds', 0):.0f} s"
                f"{same}")
    return None


WRAPUP_MEASURES_S04 = [   # (highlight, where, reader): one row each; the text comes from the section's own values
    ("1 · session → trace → span, on AgentCore Runtime", "§1.1", wrapup_runtime_s04),
    ("2 · the tool failed, the request succeeded", "§1.2", wrapup_failure_s04),
    ("1 · a local agent, one pipeline, three places", "§1.3", wrapup_local_s04),
    ("1 · head sampling vs always-on", "§1.4", wrapup_sampling_s04),
    ("3 · a looping agent: no error, more tokens", "§2.1", wrapup_loop_s04),
    ("3 · loop detection + cost protection alarms", "§2.2", wrapup_alarms_s04),
    ("4 · Nova judges on demand", "§3.2, §3.3", wrapup_on_demand_s04),
    ("4 · online evaluation", "§3.4", wrapup_online_s04),
]


WRAPUP_RAN_S04 = {"§1": "S01_RUNTIME_OK", "§2": "S02_ALARMS_UP", "§3": "S03_EVALS_ON"}   # bound by each section's 1st cell


def wrapup_measured_s04() -> pd.DataFrame:
    rows = []
    for highlight, where, read in WRAPUP_MEASURES_S04:
        sec = where[:2]
        missing = (f"— not live in this run: {sec} printed a ⚠ line and showed recorded results"
                   if WRAPUP_RAN_S04[sec] in globals() else f"— {sec} did not run in this kernel")
        try:
            text = read() or missing
        except Exception as e:  # noqa: BLE001 — one unreadable row must never stop the wrap-up (the cleanup runs)
            text = f"⚠ could not read it ({type(e).__name__})"
        rows.append({"highlight": highlight, "§": where, "measured in this run": text})
    return pd.DataFrame(rows)


def wrapup_show_measured_s04(table: pd.DataFrame) -> None:
    display(table.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "normal"})
            .set_properties(subset=["measured in this run"], **{"min-width": "520px"})
            .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))


def wrapup_component_s04(led: pd.DataFrame) -> pd.Series:
    """Each ledger row's part of the bill: agents / judges (Nova tokens) / fees / runtime / cloudwatch / other."""
    svc = led["model"].eq("AgentCore")
    name = led["agent"].astype(str).str.split(" (", regex=False).str[0]
    part = pd.Series("other", index=led.index)
    part[~svc] = "agents"
    part[~svc & led["agent"].astype(str).str.startswith("judge ")] = "judges"
    part[svc & name.eq("evaluation_custom")] = "fees"
    part[svc & name.str.endswith("_judge_tokens_est")] = "judges"
    part[svc & name.str.startswith("runtime_")] = "runtime"
    part[svc & name.str.startswith("cloudwatch_")] = "cloudwatch"
    return part


def wrapup_usd_s04(v: float, tex: bool = False) -> str:
    """$ with two significant digits (tex=True escapes the $ for matplotlib)."""
    places = max(2, 1 - math.floor(math.log10(abs(v)))) if v else 2
    return ("\\$" if tex else "$") + f"{v:.{places}f}"


WRAPUP_ITEMS_S04 = {   # service -> what the bill line says
    "evaluation_custom": "AgentCore Evaluations: evaluations (fee each)",
    "online_judge_tokens_est": "Nova judge tokens of the online evaluations (estimate)",
    "batch_judge_tokens_est": "Nova judge tokens of the batch evaluation (estimate)",
    "runtime_vcpu_hour": "AgentCore Runtime vCPU-hours (estimate)",
    "runtime_gb_hour": "AgentCore Runtime GB-hours (estimate)",
    "cloudwatch_ingest_gb": "CloudWatch Logs ingestion, GB (spans + runtime logs; estimate)",
    "cloudwatch_alarm_highres_month": "CloudWatch high-resolution alarms, alarm-months (prorated)",
    "cloudwatch_custom_metric_month": "CloudWatch custom metrics, metric-months (prorated)",
    "cloudwatch_api_request": "CloudWatch API requests (PutMetricData, §2.3's reads)",
    "cloudwatch_insights_scan_gb": "CloudWatch Logs Insights, GB scanned"}


def wrapup_bill_s04() -> pd.DataFrame:
    """Every ledger row of this run, one line per billed item. Asserts that the lines add up to the whole ledger."""
    led = LEDGER.df()
    led = led.reindex(columns=list(dict.fromkeys([*led.columns, "model", "agent", "scope", "cost_usd", "input_tokens",
                                                  "output_tokens", "service_units"])))
    led["part"] = wrapup_component_s04(led)
    led["section"] = led["scope"].astype(str).str.extract(r"^(§\d+)")[0].fillna("?")
    svc = led["model"].eq("AgentCore")
    names = {v: " ".join(w.capitalize() for w in k.split("-")) for k, v in mc.MODELS.items()}   # 'Nova 2 Lite'
    plain = lambda v: f"{v:,.6f}".rstrip("0").rstrip(".")    # noqa: E731 — 0.0895, 0.00001 (never 1e-05)
    booked = {r["service"]: r["unit_price"] for r in WRAPUP_SERVICE_ROWS_S04}   # §4's estimates carry their own price
    out = []
    for part, who in (("agents", "the agents"), ("judges", "the judges, on demand (measured)")):
        m = led[~svc & led["part"].eq(part)]
        for model, g in m.groupby("model"):
            price = mc.PRICES_PER_1M.get(model, (0, 0, 0))
            out.append({"part": part, "item": f"{names.get(model, model)} tokens: {who}",
                        "units": f"{int(g['input_tokens'].sum()):,} in + {int(g['output_tokens'].sum()):,} out "
                                 f"({len(g)} call{'s' * (len(g) != 1)})",
                        "unit price": f"${price[0]} in / ${price[1]} out per 1M", "US$": float(g["cost_usd"].sum()),
                        "booked in": ", ".join(sorted(g["section"].unique()))})
    for service, g in led[svc].groupby(led.loc[svc, "agent"].astype(str).str.split(" (", regex=False).str[0]):
        units = float(pd.to_numeric(g["service_units"], errors="coerce").fillna(0).sum())
        cost = float(g["cost_usd"].sum())
        out.append({"part": g["part"].iloc[0], "item": WRAPUP_ITEMS_S04.get(service, service),
                    "units": f"{units:,.0f}" if units >= 10 else plain(units),
                    "unit price": "$" + plain(mc.AGENTCORE_PRICES.get(service, booked.get(service, 0.0))), "US$": cost,
                    "booked in": ", ".join(sorted(g["section"].unique()))})
    bill = pd.DataFrame(out, columns=["part", "item", "units", "unit price", "US$", "booked in"])
    assert abs(bill["US$"].sum() - float(led["cost_usd"].sum())) < 1e-9, "a ledger row is missing from the bill"
    order = {k: i for i, k in enumerate(viz.COST_COLORS)}
    return bill.sort_values(["part", "US$"], key=lambda col: col.map(order) if col.name == "part" else -col) \
        .reset_index(drop=True)


def plot_bill_s04(bill: pd.DataFrame) -> dict:
    """ONE chart: this run's cost by part of the bill (colors = viz.COST_COLORS); the title states each part's share."""
    parts = bill.groupby("part")["US$"].sum().reindex(list(viz.COST_COLORS)).dropna()
    parts = parts[parts > 0]
    total = float(parts.sum())
    if not total:
        print("Nothing billed in this kernel yet: no chart.")
        return {"total": 0.0}
    led = LEDGER.df()
    agents = led[led["model"].ne("AgentCore") & ~led["agent"].astype(str).str.startswith("judge ")]
    by_sec = agents.groupby(agents["scope"].astype(str).str.extract(r"^(§\d+)")[0])["cost_usd"].sum()
    shares = {"evaluations": parts.get("fees", 0.0) + parts.get("judges", 0.0), "agents": parts.get("agents", 0.0),
              "Runtime": parts.get("runtime", 0.0), "CloudWatch": parts.get("cloudwatch", 0.0),
              "other fees": parts.get("other", 0.0)}
    biggest = sorted(((v / total, k) for k, v in shares.items() if v > 0), reverse=True)[:2]   # fits 10.5 in
    what = {"evaluations": " (fees + judge tokens)", "Runtime": " (estimate)"}
    est = bill.loc[bill["part"].eq("judges") & bill["item"].str.contains("(estimate)", regex=False), "US$"].sum()
    labels = {k: viz.COST_LABELS[k] for k in parts.index}   # the online/batch judge tokens are estimates: say so
    if "judges" in labels and est > 0:
        labels["judges"] += " (estimate)" if est >= parts["judges"] - 1e-12 else " (partly estimated)"
    fig, ax = plt.subplots(figsize=(10, 0.55 * len(parts) + 1.9))
    viz.hbar([labels[k] for k in parts.index], list(parts.values),
             colors=[viz.COST_COLORS[k] for k in parts.index], fmt="\\${:.4f}", ax=ax,
             xlabel="US\\$ this run (ledger: every model call + §4's time- and volume-based charges)",
             title=f"This run so far: {wrapup_usd_s04(total, tex=True)}: "
                   + ", ".join(f"{k} {v:.0%}{what.get(k, '')}" for v, k in biggest),
             subtitle="The agents' Nova tokens by section: " + " · ".join(
                 f"{s} {wrapup_usd_s04(v, tex=True)}" for s, v in by_sec.items() if v > 0))
    fig.tight_layout()
    plt.show()
    return {"total": total, **{k: float(v) for k, v in parts.items()}}

### 4.1 What we measured
One line per highlight, from the values each section computed and checked.

In [ ]:
WRAPUP_MEASURED_S04 = wrapup_measured_s04()          # one row per highlight, from the section's own checked values
wrapup_show_measured_s04(WRAPUP_MEASURED_S04)

**What to notice.** The failed tool, the loop and the missing disclaimer all came back without an error. Slide 55's point:
an HTTP 200 from an agent does not prove that it did its job. Observability shows what the agent did (§1,
§2); evaluation shows whether it did it well (§3). You need both.

<details><summary>Where this demo sits on slide 6's maturity stages and slide 49's cycle (click to open)</summary>

| Slide 6 stage | In this demo |
|---|---|
| 1 Foundational: collecting telemetry data | ADOT on AgentCore Runtime with no tracing code, one OpenTelemetry pipeline for the local agents (§1.1, §1.3) |
| 2 Intermediate: telemetry analysis and insights | spans read back with Logs Insights, the waterfall, the per-trace token table (§1.1, §1.2) |
| 3 Advanced: correlation and anomaly detection | one trace id across metric, log and trace (§1.2); the alarms are static thresholds, anomaly detection had to train first (§2.2) |
| 4 Proactive: automatic root cause identification | the online evaluation of §3.4: every session scored, each score explained |

Slide 49's improvement cycle on the same agent: **baseline** = on-demand evaluation before release (§3.2, §3.3),
**monitor** = online evaluation plus alarms (§3.4, §2.2), **insight** = the judge's explanation of a NonCompliant
reply, **optimize** = the `coach_rule` prompt, re-scored. Each failure you find becomes one more test case.
</details>

### 4.2 What this run cost
Every model call and every on-demand evaluation became a ledger row (§0–§3). §4 added Runtime hours (estimated from list
prices), online or batch evaluations with their estimated judge tokens, the alarms and metrics (prorated), and
CloudWatch ingestion and queries.

In [ ]:
WRAPUP_BILL_S04 = wrapup_bill_s04()
WRAPUP_PARTS_S04 = plot_bill_s04(WRAPUP_BILL_S04)
if WRAPUP_PARTS_S04["total"]:
    _share = {k: v / WRAPUP_PARTS_S04["total"] for k, v in WRAPUP_PARTS_S04.items() if k != "total"}
    print(f"This run so far: ${WRAPUP_PARTS_S04['total']:.4f} over {len(LEDGER.df())} ledger rows · evaluations "
          f"{_share.get('fees', 0) + _share.get('judges', 0):.0%} (fees {_share.get('fees', 0):.0%} + judge tokens "
          f"{_share.get('judges', 0):.0%}) · agents {_share.get('agents', 0):.0%} · Runtime hosting "
          f"{_share.get('runtime', 0):.0%} (estimate) · CloudWatch, which carried every span, metric and alarm: "
          f"{_share.get('cloudwatch', 0):.1%}")
display(HTML("<details><summary>This run's bill, line by line: every ledger row (click to open)</summary>"
             + WRAPUP_BILL_S04.drop(columns="part").style.hide(axis="index").format({"US$": "${:.6f}"}).to_html()
             + "<p>Not metered, each under $0.001: S3 storage of the agent's code zip, X-Ray span indexing (~1 %), "
               "CloudWatch reads other than §2.3's (GetMetricData, FilterLogEvents), IAM (free).</p></details>"))

**What to notice.** Evaluation cost follows judge model × evaluators × sampling rate (slide 56): a fee plus the
judge's tokens per evaluation. Our online configuration scores 100 % of sessions; in production, start small.

### 4.3 Cleanup, verified by id
**While this runs (≤ 90 s):** read slide 55's objectives below. Each resource is checked **by id**; only what
is not `gone ✓` is listed, with the command that finishes it. Spans in `aws/spans` and the custom metrics stay (they
cannot be deleted; \$0 once idle).

In [ ]:
def wrapup_show_verify_s04(verify: dict) -> None:
    """The by-id verdicts: only the rows that are NOT gone on screen, every row one click away."""
    table = pd.DataFrame([{"status": s, "resource": r} for r, s in verify.items()], columns=["status", "resource"])
    still_there = table[~table["status"].str.startswith("gone")]
    if not still_there.empty:
        display(still_there.style.hide(axis="index"))
    display(HTML(f"<details><summary>All {len(table)} resources, checked by id (click to open)</summary>"
                 + table.to_html(index=False) + "</details>"))

In [ ]:
if not WRAPUP_STEPS_S04:
    print(f"RUN_ID {RUN_ID} is a shared dev stack: nothing was deleted and nothing needs checking "
          f"({len(observability_resources())} resources kept for the builders).")
else:
    _joined, _value, _waited = mc.wait_until(lambda: WRAPUP_CLEANUP_S04.join(timeout=0.1), max_wait=90, poll=3,
                                             label="the background cleanup",
                                             describe=lambda _v: f"{len(OBS_CLEANUP_STATUS)} cleanup results so far")
    WRAPUP_VERIFY_S04 = observability_verify_gone()           # get_* by id -> NotFound / DELETING, then a name sweep
    wrapup_show_verify_s04(WRAPUP_VERIFY_S04)
    for _problem in WRAPUP_CLEANUP_S04.problems:
        print("  problem:", mc.short(mc.mask_account(_problem, ACCOUNT), 160))
    _left = [r for r, s in WRAPUP_VERIFY_S04.items() if s.startswith("NOT CLEANED")]
    print(f"{len(WRAPUP_VERIFY_S04) - len(_left)} gone ✓ · {len(_left)} NOT CLEANED · "
          f"{time.time() - WRAPUP_T0_S04:.0f} s since the cleanup started")
    if not _joined:
        print("The cleanup is still running: re-run this cell in a minute.")
    if _left or WRAPUP_CLEANUP_S04.problems:
        print(f"NOT CLEANED: {len(_left)} resource(s). Finish from a terminal in M04_Observability/:\n"
              f"   python teardown_m04.py --run-id {RUN_ID} --yes")
    elif _joined:
        print(f"All resources of RUN_ID {RUN_ID} are gone, each checked by id.")

**Module summary (slide 55)**

| Objective | Where you did it |
|---|---|
| Trace and monitor agents end to end, across several agents | §1 session → trace → span, §2 alarms |
| Build a way to evaluate agents, before and after release | §3 derived and custom Nova judges, on demand and online |
| Send agent telemetry to the observability tools a company already runs | §1.3 one OTLP pipeline, three destinations |

### Discussion prompts
<details><summary>More questions (click to open)</summary>

- Slide 6: which maturity stage is your organization at today, and which one capability would move it up a stage?
- Slide 49: which failure from this demo would you turn into a test case first, and which evaluator would score it?
- Slide 56: what would the online evaluation of §3.4 cost per day for 50,000 conversations at 10 % sampling, with
  three evaluators? (Use the fee and judge-token lines of 4.2.)
</details>